# Crushing Grapes, Crunching Numbers - EDA & Sentiment

# 1.0 Introduzione al progetto


## 1.1 Dataset Utilizzato

 Il presente report fornisce un’approfondita ricerca sul dataset di vini Wine Enthusiast (circa 130k recensioni) contenente colonne quali:

| Nome colonna            | Esempio                            | dtype   | Note utili                              |
| ----------------------- | ---------------------------------- | ------- | --------------------------------------- |
| `record_id`             | 0                                  | int64   | Indice numerico aggiunto dopo il load   |
| `country`               | *US*                               | object  | ISO/english country names               |
| `description`           | *“Gutsy aromas of blackberry and…* | object  | Testo free-form, 10-150 parole          |
| `designation`           | *Reserve*                          | object  | Nome cuvée/linea; molti NaN             |
| `points`                | 87                                 | int64   | Punteggio da 80 a 100                   |
| `price`                 | 15.0                               | float64 | In dollari USA                          |
| `province`              | *California*                       | object  | Regione amministrativa o AVA principale |
| `region_1`              | *Napa Valley*                      | object  | Sotto-zona / appellation                |
| `region_2`              | *Napa*                             | object  | Macro-area più ampia; spesso vuota      |
| `taster_name`           | *Roger Voss*                       | object  | Nome del degustatore                    |
| `taster_twitter_handle` | *@vossroger*                       | object  | Handle Twitter; alcuni shared account   |
| `title`                 | *Blackberry & Pinot Noir 2013*     | object  | Titolo completo dell’articolo           |
| `variety`               | *Pinot Noir*                       | object  | Vitigno principale                      |
| `winery`                | *Robert Mondavi*                   | object  | Produttore/cantina                      |

 
Di seguito troverai informazioni strutturate sui principali vitigni, sui profili vinicoli delle regioni e nazioni presenti nel dataset, sul sistema di punteggio utilizzato dalle riviste specializzate, sulle fasce di prezzo del mercato vinicolo e sui trend osservabili tra prezzo e qualità (punteggio).



## 1.1 Librerie Utilizzate

| Ambito                            | Pacchetto                                         |
| --------------------------------- | ------------------------------------------------- |
| Moduli Libreria Standard          | **collections**, **re**, **typing**, **warnings** |
| Calcolo numerico & data wrangling | **numpy**, **pandas**                             |
| Grafici & dashboard               | **matplotlib**, **seaborn**, **plotly**           |
| Machine Learning                  | **scikit-learn** (`sklearn`), **torch** (PyTorch) |
| NLP & modelli pre-addestrati      | **transformers** (Hugging Face)                   |
| Elaborazione linguistica classica | **nltk** (con `stopwords`)                        |
| Visualizzazione testo             | **wordcloud**                                     |
| Utility stringhe                  | **unidecode**                                     |


## 1.2 Chi sono

Visita il mio [blog personale](https://stefanonocco.it/) o colleghiamoci su:  
- [Kaggle](https://www.kaggle.com/snocco)
- [LinkedIn](https://www.linkedin.com/in/stefanonocco/)  
- [Instagram](https://www.instagram.com/steno.py/)  

***Stefano Nocco***

# 2.0 Caricamento dei Dati e Ispezione Iniziale dei Dati


In [ ]:
# Inizio notebook: src/data_processing.py o direttamente in eda.ipynb

import pandas as pd
from typing import Any, Optional

import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
FILEPATH: str = "winemag-data-130k-v2.csv"

# Carica tutte le colonne, senza downcasting iniziale
df_raw: pd.DataFrame = pd.read_csv(
    FILEPATH,
    low_memory=False,   # evita warning su tipi misti
    na_values=["", "NA", "n/a"]  # personalizza se serve
)

# Imposto come indice la colonna numerica iniziale
if "Unnamed: 0" in df_raw.columns:
    df_raw.rename(columns={"Unnamed: 0": "record_id"}, inplace=True)
    df_raw.set_index("record_id", inplace=True)

print(f"Dataset loaded:\n{df_raw.shape[0]} righe × {df_raw.shape[1]} colonne")


In [ ]:
# Prime 5 righe
display(df_raw.head())

In [ ]:
# Ultime 5 righe
display(df_raw.tail())

In [ ]:
# Tipi e non-null count
df_raw.info(verbose=True)

In [ ]:
# Statistiche descrittive (numeriche e categoriche)
df_raw.describe(include="all").T

Il fatto che features come "description", che, per ipotesi dovrebbero essere uniche per ogni vino, mostrino una frequenza di comparsa > 1, mi fa pensare che ci siano dei valori duplicati. Verifichiamo.

# 3.0 Data Cleaning

## 3.1 Rimozione dei duplicati

In [ ]:
# Copio il dataset per mantenere l'originale inalterato
df_clean = df_raw.copy()

# Unicità dell'ID
assert df_clean.index.is_unique, "record_id non è univoco!"

# Range 'points'
assert df_clean["points"].between(0, 100).all(), "Valori di points fuori range atteso!"

# Quante righe sono duplicate su TUTTE le colonne?
# Summary before deduplication
shape_before = df_clean.shape
duplicate_rows = df_clean.duplicated().sum()

# Remove fully identical duplicate rows
df_clean = df_clean.drop_duplicates()

# Summary after deduplication
shape_after = df_clean.shape
duplicates_removed = duplicate_rows  # number of rows dropped


print(f"Shape before deduplication              : {shape_before}")
print(f"Number of fully identical duplicate rows: {duplicates_removed}")
print(f"Shape after deduplication               : {shape_after}")


Ho rimosso 9983 valori duplicati. Rifaccio un describe per osservare le differenze.

In [ ]:
df_clean.describe(include="all").T

## 3.2 Descrizioni duplicate?

Non mi aspettavo questo risultato, perché vedo che in "description" abbiamo ancora una frequenza > 1. 

Saranno gli stessi vini? Potrebbero essere delle recensioni identiche da parte del/dei sommelier?  

I valori duplicati dovrebbero essere 33 (count - unique = 33)

Verifico subito.

In [ ]:
# Identifico record dove la descrizione è duplicata
dup_rows = df_clean[df_clean.duplicated(subset="description", keep=False)].copy()
dup_rows = dup_rows.sort_values('description')

In [ ]:
dup_rows.head(6)

Allora, dobbiamo fare dei ragionamenti differenti a seconda dei record analizzati. Avendo già rimosso i duplicati, so per certo che non sono record identici. In alcuni casi si tratta di vini differenti di differenti cantine pertanto devo mantenere questi record. Anche i voti sono differenti.  
Verifico se i record hanno valori identici nelle colonne chiave.

In [ ]:
key_cols = ["description", "title", "winery",
            "designation", "points", "price"]

dup_rows_to_remove = df_clean[df_clean.duplicated(subset=key_cols, keep=False)].copy()

In [ ]:
dup_rows_to_remove

Ok, mi restituisce un dataset vuoto, non rimuoverò i valori. 

N.B. Ho verificato anche manualmente: in effetti le recensioni identiche sono riferite a vini/cantine/località differenti.

## 3.3 Analisi dei valori mancanti

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Optional

def report_missing_values(
    df: pd.DataFrame,
    *,
    top_n: Optional[int] = 10,
    show_heatmap: bool = True,
    figsize: tuple[int, int] = (12, 6),
) -> pd.Series:
    """
    Analyze and visualize missing values in a DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame to analyze.
    top_n : int | None, default 10
        Number of columns to show in the percentage ranking (None → all).
    show_heatmap : bool, default True
        If True, draws a heatmap of the missing cells.
    figsize : tuple[int, int], default (12, 6)
        Figure size for the heatmap.
    
    Returns
    -------
    pd.Series
        Series with the count of NaNs per column (only those with at least
        one missing value), sorted in descending order.
    """
    # 1. Conteggio dei NaN
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

    print("Missing values per colonna:")
    print(missing_counts, "\n")

    # 2. Percentuali di NaN
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    if top_n is not None:
        display(missing_pct.head(top_n))
    else:
        display(missing_pct)

    # 3. Heatmap opzionale
    if show_heatmap:
        plt.figure(figsize=figsize)
        sns.heatmap(df.isna(), cbar=False, yticklabels=False)
        plt.title("Mappa dei valori mancanti")
        plt.show()

    return missing_counts

In [ ]:
missing_counts = report_missing_values(df_clean, top_n=None, show_heatmap=True)

## 3.4 Esploriamo le features

Risolto il problema delle recensioni duplicate, ora possiamo affermare che abbiamo:

- 43 differenti stati
- 707 varietà di vini
- 19 Sommelier che hanno recensito i vini

Vediamo i valori e la loro densità del dataset. Creo due funzioni:
- una per fare un sommario, in modo da osservare meglio i valori e la loro densità nel dataset.
- l'altra che mi rappresenti graficamente le features.

In questo modo ho sempre una visione numerica e una grafica dei dati

In [ ]:
def summarize_feature(
    df: pd.DataFrame,
    column: str,
    /,
    *,
    dropna: bool = False,
    percent_decimals: int = 3
) -> pd.DataFrame:
    """
    Return a summary table (value, count, percent) for a single categorical column.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame.
    column : str
        Name of the column to summarise.
    dropna : bool, default False
        If True, exclude NaN from the counts.
    percent_decimals : int, default 2
        Number of decimal places for the percentage values.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - 'value'   : distinct values of the feature (as string)
        - 'count'   : absolute frequency
        - 'percent' : relative frequency in %
        Sorted in descending order by count.
    """
    
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame.")

    total_rows: int = len(df)
    vc = df[column].value_counts(dropna=dropna)

    summary = (
        pd.DataFrame(
            {
                "value": vc.index.astype(str),
                "count": vc.values,
            }
        )
        .assign(percent=lambda d: (d["count"] / total_rows * 100).round(percent_decimals))
        .sort_values(by="count", ascending=False)
        .reset_index(drop=True)
    )

    return summary

In [ ]:
import pandas as pd
import plotly.express as px
from plotly.graph_objects import Figure
from typing import Optional

def plot_feature_distribution_interactive(
    df: pd.DataFrame,
    column: str,
    /,
    *,
    top_n: Optional[int] = 20,
    bins: int | None = None,
    title_prefix: str = "Distribuzione di",
    height: int = 500,
    width: int = 900,
) -> Figure:
    """
    Interactive distribution plot for a single feature using Plotly.

    Numeric  -> histogram (nbins=bins).  
    Category -> horizontal bar chart of most common categories.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame.
    column : str
        Feature to plot.
    top_n : int | None, default 20
        For categorical features: number of most frequent categories shown.
        If None, plot all categories (not recommended for high cardinality).
    bins : int | None, default None
        Number of bins for numeric histogram. If None, Plotly picks automatically.
    title_prefix : str, default "Distribuzione di"
        Prefix for the plot title.
    height, width : int
        Figure dimensions in pixels.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame.")

    # Numeric → histogram
    if pd.api.types.is_numeric_dtype(df[column]):
        fig = px.histogram(
            df, 
            x=column, 
            nbins=bins,
            title=f"{title_prefix} {column}",
        )
        fig.update_yaxes(title="Frequency")

    else:
        # Categorical → barplot (top_n)
        vc = df[column].value_counts(dropna=False)
        if top_n is not None:
            vc = vc.head(top_n)

        cat_df = vc.reset_index()
        cat_df.columns = [column, "count"]

        fig = px.bar(
            cat_df[::-1],  # reverse for descending order top-to-bottom
            x="count",
            y=column,
            orientation="h",
            title=f"{title_prefix} {column}",
        )

    fig.update_layout(height=height, width=width, bargap=0.1)
    return fig

### 3.4.1 Stati

In [ ]:
summarize_feature(df_clean,"country")

#### Record senza informazioni sullo stato

N.B. 59 record sono privi di stato. Vediamoli:

In [ ]:
missingCountries = df_clean[df_clean['country'].isna()]

In [ ]:
missingCountries

In seguito all'analisi dei valori mancanti ho riscontrato che ci sono 59 record privi di Stato e relative regioni di catalogazione. 
Basandoci su un dataset di recensioni di oltre 100K righe possiamo scegliere due strade:
- Rimuovere i 59 record
- Estrarre il nome dello stato basandoci sulla Cantina (winery) e riempire questi record vuoti

Ad una prima impressione, la prima scelta potrebbe essere la più comoda, ma voglio valutare informazioni quali punteggi su Wine Enthusiast e relativi prezzi. 

Inoltre, potrebbero appartenere a stati con pochi record, pertanto potrebbero essere essenziali per successive analisi.
Vediamo se sin tratta di 59 diverse cantine oppure se c'è qualcosa in comune.

In [ ]:
summarize_feature(missingCountries, "winery")

Bene, abbiamo 26 differenti cantine senza stato. Ora faccio un po' di ricerca e in seguito farò il mapping.

| #  | Cantina                         | Country                | Province / Region (or equivalent)                                           |
| -- | ------------------------------- | ---------------------- | --------------------------------------------------------------------------- |
| 0  | Neumeister                      | **Austria**            | Styria – Vulkanland Steiermark, municipality of Straden                     |
| 1  | Orbelus                         | **Bulgaria**           | Blagoevgrad Province – Melnik / Kresna area                                 |
| 2  | Stone Castle                    | **Kosovo**             | Rahovec Municipality (Gjakova District)                                     |
| 3  | Psagot                          | **Israel / West Bank** | Binyamin Region, Jerusalem Mountains                                        |
| 4  | Ross-idi                        | **Bulgaria**           | Sliven Province                                                             |
| 5  | El Capricho                     | **Uruguay**            | Durazno Department – Villa del Carmen                                       |
| 6  | Gotsa Family Wines              | **Georgia**            | Kvemo Kartli – village Kiketi (Tbilisi Municipality)                        |
| 7  | Tsililis                        | **Greece**             | Trikala Regional Unit, Thessaly                                             |
| 8  | St. Donat                       | **Hungary**            | Veszprém County – Csopak, Balatonfüred-Csopak                               |
| 9  | Ktima Voyatzi                   | **Greece**             | Velventos, Kozani Regional Unit, Western Macedonia                          |
| 10 | Les Frères Dutruy               | **Switzerland**        | Canton of Vaud – La Côte / Terre Sainte                                     |
| 11 | Barton & Guestier               | **France**             | Gironde Department, Nouvelle-Aquitaine – Blanquefort (Bordeaux)             |
| 12 | Teliani Valley                  | **Georgia**            | Kakheti Region – Telavi area                                                |
| 13 | Bartho Eksteen                  | **South Africa**       | Western Cape – Hemel-en-Aarde Valley, Hermanus                              |
| 14 | Undurraga                       | **Chile**              | Santiago Metropolitan Region, Talagante Province – Maipo Valley (Santa Ana) |
| 15 | Mt. Beautiful                   | **New Zealand**        | North Canterbury – Hurunui District (Spotswood)                             |
| 16 | Bachelder                       | **Canada**             | Ontario – Beamsville, Niagara Peninsula                                     |
| 17 | Chilcas                         | **Chile**              | Maule Region – Maule Valley                                                 |
| 18 | Santa Ema                       | **Chile**              | Santiago Metropolitan Region – Maipo Valley (Isla de Maipo)                 |
| 19 | Amiran Vepkhvadze               | **Georgia**            | Imereti Region – Zestafoni                                                  |
| 20 | Familia Deicas                  | **Uruguay**            | Canelones Department – Juanicó (Las Violetas)                               |
| 21 | Lismore                         | **South Africa**       | Western Cape – Greyton, Cape South Coast                                    |
| 22 | Logodaj                         | **Bulgaria**           | Blagoevgrad Province – Struma Valley, Gen. Todorov                          |
| 23 | Somlói Vándor                   | **Hungary**            | Veszprém County – Somló Hill (Nagy Somló)                                   |
| 24 | Kakhetia Traditional Winemaking | **Georgia**            | Guria Region – village Askana (historic site); HQ in Tbilisi                |
| 25 | Newton Johnson                  | **South Africa**       | Western Cape – Upper Hemel-en-Aarde, Hermanus                               |
| 26 | Büyülübağ                       | **Turkey**             | Balıkesir Province – Avşa Island, Sea of Marmara                            |


Che dire, ho trovato vini georgiani, bulgari, kosovari. E' stata un'operazione corretta da effettuare, in quanto ho trovato anche stati poco rappresentati nel dataset.
Ora preparo un mapping per correggere i 59 record con valori mancanti nello stato.

In [ ]:
import pandas as pd
from typing import Dict, Tuple

# --------------------------------------------------------------------
# Dizionario di mapping cantine → (country, province/region)
# --------------------------------------------------------------------
WINERY_MAP: Dict[str, Tuple[str, str]] = {
    "Neumeister": ("Austria", "Styria – Vulkanland Steiermark, municipality of Straden"),
    "Orbelus": ("Bulgaria", "Blagoevgrad Province – Melnik / Kresna area"),
    "Stone Castle": ("Kosovo", "Rahovec Municipality (Gjakova District)"),
    "Psagot": ("Israel", "Binyamin Region, Jerusalem Mountains"),
    "Ross-idi": ("Bulgaria", "Sliven Province"),
    "El Capricho": ("Uruguay", "Durazno Department – Villa del Carmen"),
    "Gotsa Family Wines": ("Georgia", "Kvemo Kartli – village Kiketi (Tbilisi Municipality)"),
    "Tsililis": ("Greece", "Trikala Regional Unit, Thessaly"),
    "St. Donat": ("Hungary", "Veszprém County – Csopak, Balatonfüred-Csopak"),
    "Ktima Voyatzi": ("Greece", "Velventos, Kozani Regional Unit, Western Macedonia"),
    "Les Frères Dutruy": ("Switzerland", "Canton of Vaud – La Côte / Terre Sainte"),
    "Barton & Guestier": ("France", "Blanquefort (Bordeaux)"),
    "Teliani Valley": ("Georgia", "Kakheti Region – Telavi area"),
    "Bartho Eksteen": ("South Africa", "Western Cape – Hemel-en-Aarde Valley, Hermanus"),
    "Undurraga": ("Chile", "Santiago Metropolitan Region, Talagante Province – Maipo Valley (Santa Ana)"),
    "Mt. Beautiful": ("New Zealand", "North Canterbury – Hurunui District (Spotswood)"),
    "Bachelder": ("Canada", "Ontario – Beamsville, Niagara Peninsula"),
    "Chilcas": ("Chile", "Maule Region – Maule Valley"),
    "Santa Ema": ("Chile", "Santiago Metropolitan Region – Maipo Valley (Isla de Maipo)"),
    "Amiran Vepkhvadze": ("Georgia", "Imereti Region – Zestafoni"),
    "Familia Deicas": ("Uruguay", "Canelones Department – Juanicó (Las Violetas)"),
    "Lismore": ("South Africa", "Western Cape – Greyton, Cape South Coast"),
    "Logodaj": ("Bulgaria", "Blagoevgrad Province – Struma Valley, Gen. Todorov"),
    "Somlói Vándor": ("Hungary", "Veszprém County – Somló Hill (Nagy Somló)"),
    "Kakhetia Traditional Winemaking": ("Georgia", "Guria Region – village Askana (historic site); HQ in Tbilisi"),
    "Newton Johnson": ("South Africa", "Western Cape – Upper Hemel-en-Aarde, Hermanus"),
    "Büyülübağ": ("Turkey", "Balıkesir Province – Avşa Island, Sea of Marmara"),
}

# --------------------------------------------------------------------
# Funzione di mapping
# --------------------------------------------------------------------
def apply_winery_mapping(
    df: pd.DataFrame,
    mapping: Dict[str, Tuple[str, str]],
    *,
    winery_col: str = "winery",
    country_col: str = "country",
    province_col: str = "province",
    inplace: bool = False,
) -> pd.DataFrame:
    """
    Riempie `country` e `province` per i record con valori mancanti,
    utilizzando un mapping cantina → (country, province).

    Parameters
    ----------
    df : pd.DataFrame
        Il DataFrame da aggiornare.
    mapping : dict[str, tuple[str, str]]
        Dizionario {cantina: (country, province)}.
    winery_col : str, default "winery"
        Nome della colonna con il nome della cantina.
    country_col : str, default "country"
        Colonna da popolare con lo stato.
    province_col : str, default "province"
        Colonna da popolare con la provincia/regione.
    inplace : bool, default False
        Se True, modifica `df` in-place e restituisce lo stesso oggetto.
        Se False, restituisce una *copia* con le modifiche.

    Returns
    -------
    pd.DataFrame
        DataFrame con i valori mappati.
    """
    target_df = df if inplace else df.copy()

    # normalizziamo i nomi per confronti case-insensitive
    norm_map = {k.lower().strip(): v for k, v in mapping.items()}

    mask_missing = target_df[country_col].isna() & target_df[province_col].isna()

    # iteriamo solo sulle righe interessate
    for idx, winery in target_df.loc[mask_missing, winery_col].items():
        if pd.isna(winery):
            continue
        key = str(winery).lower().strip()
        if key in norm_map:
            country, province = norm_map[key]
            target_df.at[idx, country_col] = country
            target_df.at[idx, province_col] = province

    return target_df

In [ ]:
# test della funzione sulla porzione di dati privi di paesi e province
missingCountries = apply_winery_mapping(missingCountries, WINERY_MAP, inplace=True)

In [ ]:
missingCountries

In [ ]:
df_clean = apply_winery_mapping(df_clean, WINERY_MAP, inplace=True)

In [ ]:
# Verifico che il mapping sia corretto -> OK!
missing_counts = report_missing_values(df_clean, top_n=None, show_heatmap=True)

In [ ]:
plot_feature_distribution_interactive(df_clean, "country")

### 3.4.2 Province, Region_1 & Region_2

In [ ]:
summarize_feature(df_clean, "province")[0:30]

In [ ]:
plot_feature_distribution_interactive(df_clean, "province")

#### Profili Enologici di Paesi e Regioni nel Dataset: Italia

In [ ]:
summarize_feature(df_clean[df_clean.country=='Italy'], "province")[0:30]

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'Italy'], "province")

In [ ]:
summarize_feature(df_clean[df_clean.country=='Italy'], "region_1")

L’Italia è uno dei maggiori produttori al mondo, rinomata per la diversità vinicola: ogni regione italiana coltiva uve autoctone proprie, offrendo una vasta gamma di sapori e stili. Due pilastri dell’enologia italiana sono il Piemonte e la Toscana, celebri soprattutto per i grandi vini rossi da vitigni autoctoni (Nebbiolo, Barbera, Sangiovese) e talvolta da tagli internazionali (Cabernet, Merlot).        

| Regione / Territorio             | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   |
| -------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Piemonte**                     | Eleganza e struttura. Qui nascono Barolo e Barbaresco (Nebbiolo in purezza) – vini aristocratici, tannici e longevi, soprannominati i “re” del Piemonte. Il Nebbiolo piemontese dà vini di grande finezza olfattiva (rosa, spezie) e potenza gustativa, affinati anni in botte per ammorbidirne i tannini. Altri rossi tipici: Barbera (uvaggio autoctono più morbido, acidulo e fruttato, eccellente per abbinamento gastronomico) e Dolcetto (più semplice e fresco). Da menzionare i Moscato d’Asti come aromatico bianco dolce e i vermouth storicamente prodotti a Torino. Il Piemonte vanta terroir unici (Langhe, Monferrato, Alto Piemonte) e un’enologia di tradizione antica ma innovativa.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| **Toscana**                      | Tradizione e fama internazionale. È la culla del Sangiovese, base di vini famosi come Chianti (rosso secco fruttato, tannico, con note di ciliegia e spesso affinamento in botte) e Brunello di Montalcino (Sangiovese “Grosso” in purezza, strutturato e longevo). I vini toscani rossi uniscono corpo, tannino e acidità, con profumi di violetta, ciliegia e spezie; possono essere aggraziati e rustici (Chianti) o potenti e raffinati (Brunello) a seconda della zona e dello stile di vinificazione. La Toscana è celebre anche per i “Supertuscan” – vini di taglio internazionale nati negli anni ’70, spesso da Cabernet Sauvignon, Merlot e altre uve francesi coltivate lungo la costa (es. Bolgheri) e assemblate talvolta col Sangiovese. Esempi noti: Sassicaia, Tignanello, Ornellaia, divenuti icone mondiali. La regione produce anche ottimi bianchi (es. Vernaccia di San Gimignano fragrante e sapido) e vini dolci da uve appassite (Vin Santo). La Toscana gode di un clima collinare mite ideale per la vite e paesaggi vitati di rara bellezza, motivo per cui è tra le mete predilette dai wine lovers.                                                                                                                                                                             |
| **Veneto**                       | Quantità e qualità, con stili molto diversi. In Veneto troviamo i grandi rossi della Valpolicella (Corvina, Rondinella): Amarone (vino possente ottenuto da uve appassite, con note di confettura, spezie e cioccolato), Recioto dolce e Valpolicella Ripasso (tecnica di rifermentazione sulle vinacce, che dà un vino intermedio per corpo e ricchezza). Sul fronte bianchi, il Veneto è patria del Prosecco (Glera) – spumante fresco e aromatico, leader di mercato – e del Soave (Garganega) delicato e minerale. Vanta inoltre produzioni di Pinot Grigio e altri vitigni internazionali nelle pianure. È tra le regioni italiane più produttive, unendo denominazioni industriali a vere eccellenze territoriali.                                                                                                            |
| **Altre regioni d’Italia**       | Friuli-Venezia Giulia e Trentino-Alto Adige sono rinomati per bianchi di alto livello (Friulano, Ribolla, Gewürztraminer, Müller Thurgau) e alcuni rossi eleganti (Teroldego, Schiava, Pinot Nero altoatesino). Lombardia spicca per gli spumanti metodo classico di Franciacorta (Chardonnay/Pinot) e i rossi di Valtellina (Nebbiolo). Sicilia è in forte ascesa qualitativa: accanto al celebre vino liquoroso Marsala, oggi produce rossi da Nero d’Avola (potenti e solari) e vini minerali e fini sull’Etna (da Nerello Mascalese e Carricante). Puglia e Campania: la Puglia ha tradizione di rossi caldi e morbidi da Primitivo e Negroamaro (es. Primitivo di Manduria, Salice Salentino), spesso dall’ottimo rapporto qualità/prezzo; la Campania vanta vitigni nobili come Aglianico (Taurasi, strutturato e longevo, detto “Barolo del Sud”) e bianchi tipici come Fiano e Greco di Tufo. Alto Adige e Piemonte producono anche ottimi vini dolci passiti (es. Moscato, Passito di Pantelleria in Sicilia, Recioto di Soave. L’Italia offre molteplici stili: dal Barolo austero al fresco Vermentino sardo, dal Lambrusco frizzante emiliano al Cannonau corposo. Questa ricchezza enologica rende il vino italiano celebre nel mondo per qualità e varietà pressoché ineguagliata. |


#### Profili Enologici di Paesi e Regioni nel Dataset: Francia

In [ ]:
summarize_feature(df_clean[df_clean.country=='France'], "province")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'France'], "province")

In [ ]:
summarize_feature(df_clean[df_clean.country=='France'], "region_1")

La Francia è storicamente considerata la patria del vino di qualità, con regioni vinicole leggendarie che hanno definito lo stile di molti vini. Il Paese presenta una grande diversità climatica e di terroir, eccellendo in varie tipologie (rossi, bianchi, spumanti, dolci).  
| Regione / Territorio              | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| --------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bordeaux**                      | Situata nel sud-ovest, è famosa per i suoi vini rossi da taglio bordolese (Cabernet Sauvignon, Merlot, Cabernet Franc). La regione è divisa dalla Gironda in Rive Gauche (Médoc, Graves – dominanza Cabernet Sauvignon) e Rive Droite (Saint-Émilion, Pomerol – dominanza Merlot). I migliori Bordeaux (es. Château Lafite, Latour, Margaux, Haut-Brion, Mouton) sono complessi, strutturati e longevi, con aromi di ribes nero, grafite, tabacco e spezie. Produce anche ottimi bianchi secchi (Pessac-Léognan da Sauvignon e Sémillon) e celebri vini dolci botritizzati come il Sauternes (es. Château d’Yquem).                     |
| **Borgogna (Bourgogne)**          | Nel centro-est, culla del Pinot Noir e dello Chardonnay in purezza. I vini riflettono il concetto di cru e climat: la Côte de Nuits e la Côte de Beaune offrono alcuni tra i più raffinati Pinot Noir (es. Romanée-Conti) e Chardonnay (es. Montrachet) del mondo. Produzione limitata e alta domanda alimentano la fama di cru come Gevrey-Chambertin, Vosne-Romanée, Puligny-Montrachet. A nord, lo Chablis dà Chardonnay minerali e tesi; a sud, il Beaujolais produce vini fruttati da Gamay.                                                                                                                                       |
| **Champagne**                     | Regione nel nord-est, celebre per lo spumante metodo classico omonimo. L’assemblaggio di Chardonnay, Pinot Noir e Pinot Meunier e il lungo affinamento sui lieviti conferiscono complessità (pane tostato, brioche, agrumi canditi) e perlage finissimo. Simbolo di lusso; Maison storiche (Moët & Chandon, Veuve Clicquot, Dom Pérignon, Krug) producono cuvée prestigiose. I villaggi sono classificati (Grand Cru, Premier Cru) per qualità del terroir.                                                                                                                                                                             |
| **Valle del Rodano**              | Divisa in Rodano settentrionale (clima più fresco) e meridionale. A nord il Syrah domina: Côte-Rôtie e Hermitage danno rossi profondi, pepati ed eleganti, talvolta co-fermentati con Viognier. A sud prevalgono i blend Grenache-Syrah-Mourvèdre: Châteauneuf-du-Pape è il vino simbolo, robusto, con sentori di frutta rossa macerata, erbe provenzali e liquirizia. La zona meridionale produce anche bianchi da Viognier, Roussanne, Marsanne e rosati (Tavel).                                                                                                                                                                     |
| **Loira**                         | Mosaico di sottozone lungo il fiume. Nel centro-Loira spiccano i bianchi Sancerre e Pouilly-Fumé (Sauvignon Blanc) aromatici, freschi e minerali. Nella Loira occidentale i secchi Muscadet (Melon de Bourgogne) accompagnano le ostriche. In Anjou-Touraine e Loira centrale il Chenin Blanc esprime secchi (Savennières), dolci botritici di livello mondiale (Coteaux du Layon, Quarts de Chaume) e spumanti metodo classico (Saumur, Vouvray). Ottimi anche i rossi da Cabernet Franc (Chinon, Bourgueil) e i rosati (Rosé d’Anjou).                                                                                                |
| **Alsazia**                       | Nel nord-est al confine con la Germania; clima fresco, produce quasi solo vini bianchi aromatici (Riesling, Gewürztraminer, Pinot Gris, Muscat), spesso vinificati in purezza e indicati in etichetta. Vini intensi e speziati, talvolta con residuo zuccherino; famosi i vendanges tardives e Sélection de Grains Nobles. Possiede numerosi grand cru.                                                                                                                                                                                                                                                                                 |
| **Altre regioni di Francia**      | **Provenza** è celebre per i rosati leggeri e fragranti (Côtes de Provence). **Sud-Ouest** include Cahors (Malbec tannico) e Jurançon (bianchi dolci da Petit Manseng). **Languedoc-Roussillon** produce grandi volumi di vini (IGP Pays d’Oc) con buon rapporto qualità/prezzo e rossi mediterranei da Syrah, Grenache, Carignan. **Corsica** coltiva Vermentino e Nielluccio; **Savoia/Jura** offrono vini di nicchia come il vin jaune. In generale, la Francia mantiene una reputazione d’eccellenza: i suoi vini rappresentano riferimenti stilistici internazionali, coprendo tutte le fasce dal vin de table ai cru più costosi. |


#### Profili Enologici di Paesi e Regioni nel Dataset: US

In [ ]:
summarize_feature(df_clean[df_clean.country=='US'], "province")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'US'], "province", top_n=10)

In [ ]:
summarize_feature(df_clean[df_clean.country=='US'], "region_1")

Pur essendo un produttore relativamente giovane, il Paese ha una scena vinicola vibrante e diversificata. La produzione è concentrata in California (circa 85-90 % del vino americano). Altre regioni importanti includono Washington, Oregon, New York e Virginia. L’approccio enologico esalta libertà di sperimentazione e stili innovativi, spesso ispirati a modelli europei ma con frutto più maturo, tipico dei climi caldi del “New World”.  
| Regione / Territorio                  | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     |
| ------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **California**                        | Capitale del vino USA, con oltre 4 000 aziende vinicole. Zone di punta: **Napa Valley** (Cabernet Sauvignon di grande struttura, 14-15 % vol, esempi iconici come Opus One e Screaming Eagle) e **Sonoma County** (Pinot Noir di Russian River e Carneros, Chardonnay burrosi, Zinfandel di Dry Creek, blend bordolesi di Alexander Valley). Altre aree: Central Coast (Paso Robles per Zinfandel e Syrah; Santa Barbara per Pinot Noir elegante), Sierra Foothills (vecchie vigne di Zinfandel), Central Valley (produzione di massa). Climi mediterranei e know-how fanno della California circa l’85 % della produzione USA, con vini che spaziano da entry-level a etichette premiatissime. |
| **Stato di Washington**               | Secondo produttore USA per volume. Le vigne (Columbia Valley, Yakima Valley, Walla Walla) si trovano a est della Catena delle Cascate, in clima continentale secco. Vini di riferimento: rossi bordolesi da Cabernet Sauvignon e Merlot, bianchi da Riesling e Chardonnay con buona struttura, frutto concentrato e fresca acidità. Syrah in crescita e sperimentazioni con varietà italiane; qualità media elevata e in forte sviluppo.                                                                                                                                                                                                                                                        |
| **Oregon**                            | Celebre per il Pinot Noir della **Willamette Valley**, dal clima fresco oceanico simile alla Borgogna: vini eleganti, profumi di fragolina, ciliegia e terra umida, fama mondiale nonostante rappresenti \~1 % della produzione USA. Si producono anche ottimi Chardonnay e Pinot Gris; nelle valli Umpqua e Rogue (più calde) trovano spazio varietà bordolesi e Syrah. Approccio artigianale, sostenibilità e piccole cantine.                                                                                                                                                                                                                                                                |
| **New York**                          | Terzo stato produttore di vino. I **Finger Lakes** (clima freddo moderato dai laghi) producono Riesling in stile tedesco (lime, pesca, talvolta leggero residuo zuccherino), Chardonnay freschi e spumanti metodo classico. A **Long Island** (North Fork) nascono rossi bordolesi e Chardonnay grazie al clima oceanico mite. Investimenti crescenti in bianchi aromatici e spumanti.                                                                                                                                                                                                                                                                                                          |
| **Altre zone USA**                    | **Virginia**: buoni Viognier, Bordeaux blend e varietà locali (Norton). **Texas** e altri stati del Sud: emergenti con uve resistenti al caldo (Tempranillo, Tannat). **Arizona, Colorado, New Mexico**: enclavi in altura per rossi GSM e bianchi aromatici. **Idaho, Michigan**: sperimentano con Riesling e Chardonnay in zone montane o lacustri. Gli USA contano 267 AVA riconosciute in 34 stati; Napa Valley resta la destinazione di punta per l’enoturismo e simbolo del vino americano di qualità.                                                                                                                                                                                    |


#### Profili Enologici di Paesi e Regioni nel Dataset: Spagna

In [ ]:
summarize_feature(df_clean[df_clean.country=='Spain'], "province")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'Spain'], "province")

In [ ]:
summarize_feature(df_clean[df_clean.country=='Spain'], "region_1")

Con il vigneto più esteso al mondo e una tradizione antichissima, la Spagna è nota per rossi robusti e speziati, eccellenti vini fortificati e spumanti metodo tradizionale. Il clima generalmente caldo si riflette in vini ricchi di alcol e corpo, ma nelle zone atlantiche più fresche (Galizia, Cantabrico) nascono bianchi acidi e profumati come l’Albariño di Rías Baixas. Spesso offre un ottimo rapporto qualità / prezzo.                             

| Regione / Territorio             | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| -------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Rioja**                        | Regione settentrionale patria del Tempranillo. I vini maturano a lungo in botti di rovere (spesso americano) e sono classificati in Crianza, Reserva, Gran Reserva. Un Gran Reserva maturo sprigiona aromi di frutta secca, tabacco, cuoio, vaniglia e tannini setosi; il Crianza è più fruttato e immediato. Piccole percentuali di Garnacha, Graciano e Mazuelo completano i blend. Rioja gode di reputazione internazionale per qualità a prezzi ragionevoli. |
| **Ribera del Duero**             | Entroterra castigliano, altro regno del Tempranillo (Tinto Fino). Clima continentale marcato: vini potenti, concentrati, con note di mora, prugna, cacao e tannini imponenti. Esempi di culto: Vega Sicilia Único. Grande capacità di invecchiamento.                                                                                                                                                                                                            |
| **Priorat**                      | Piccola area in Catalogna (Tarragona) famosa per rossi potentissimi da Garnacha e Cariñena coltivate su suoli scistosi (llicorella). Vini densi, alcolici, con profumi di confettura di frutti neri, liquirizia, grafite e accentuata mineralità, tannini vigorosi e prezzi tra i più alti in Spagna.                                                                                                                                                            |
| **Altri rossi spagnoli**         | Catalogna (Montsant), Aragona (Campo de Borja, Calatayud – Garnacha), Levante (Jumilla, Yecla – Monastrell), Andalusia (Syrah a Ronda), Galizia (Mencía di Bierzo e Ribeira Sacra, eleganti e floreali). Mostrano la varietà di stili dal potente Monastrell mediterraneo ai più freschi Mencía del nord-ovest.                                                                                                                                                  |
| **Cava**                         | Spumante metodo tradizionale principalmente del Penedès (Catalogna) da Macabeo, Xarel·lo, Parellada; talvolta Chardonnay/Pinot. Fresco, note di mela, agrumi e lieve lievito; versioni Reserva e Gran Reserva raggiungono alta qualità pur restando alternativa più accessibile allo Champagne.                                                                                                                                                                  |
| **Sherry (Jerez)**               | Vino fortificato di Andalusia ottenuto da uve Palomino e maturato in sistema solera. Stili secchi: Fino e Manzanilla (pallidi, mandorla, “flor”); Oloroso (ambrato, corposo, note di noci). Stili dolci: Pedro Ximénez (scurissimo, vischioso, gusto di uvetta e melassa). Patrimonio enologico di grande importanza storica.                                                                                                                                    |


#### Profili Enologici di Paesi e Regioni nel Dataset: Portogallo

In [ ]:
summarize_feature(df_clean[df_clean.country=='Portugal'], "province")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'Portugal'], "province")

Nonostante le piccole dimensioni, il Portogallo è un gigante vinicolo per tradizione e varietà di vitigni autoctoni. È celebre soprattutto per i vini fortificati (Porto e Madeira) e per rossi di forte carattere. Il Paese impiega oltre 250 vitigni locali, quasi sconosciuti altrove, e la reputazione dei suoi vini è cresciuta molto negli ultimi anni grazie all’ottimo rapporto qualità / prezzo, in particolare per i rossi del Douro, dell’Alentejo e di alcune cooperative del Dão. Il Porto rimane il simbolo nazionale, con marchi storici come Taylor’s, Graham’s e Dow’s. 

| Regione / Territorio                 | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| ------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Douro e Vinho do Porto**           | La valle del Douro, nel nord, produce l’uva per il celebre Porto. Il Porto è un vino liquoroso dolce, fortificato a metà fermentazione con acquavite vinica: stili Ruby (giovani, fruttati), Tawny (invecchiati in legno, colore ambrato e sapore di frutta secca e caramello) e Vintage (di singola annata, potente, adatto a lunghissimo invecchiamento). Uve principali: Touriga Nacional, Touriga Franca, Tinta Roriz, Tinta Barroca. Oltre al Porto, il Douro produce rossi secchi di alto livello dagli stessi vitigni: robusti, alcolici, aromatici (frutti neri maturi, spezie, cioccolato) con marcata mineralità da terreni scistosi; spesso valutati molto bene dalla critica. |
| **Dão e Bairrada**                   | Zone del Portogallo centrale: Dão (interna) e Bairrada (costiera). Il Dão offre rossi eleganti e equilibrati, spesso da Touriga Nacional in purezza o in blend, con profumi di viola, frutti di bosco e note balsamiche, buona acidità e capacità evolutiva. La Bairrada è nota per il vitigno Baga, molto tannico e acido, che dà rossi austeri bisognosi di lungo affinamento (ricordano per certi versi i Nebbiolo, con aromi di sottobosco e catrame). In Bairrada si producono anche spumanti metodo classico di interesse.                                                                                                                                                          |
| **Alentejo**                         | Ampia regione meridionale dal clima caldo e secco. Produce grandi quantità di rossi morbidi, alcolici e rotondi, spesso blend di uve locali (Aragonez / Tempranillo, Trincadeira, Alicante Bouschet, Castelão) con possibili varietà internazionali. Vini dal tannino dolce, sapori di confettura di prugne, cacao e spezie dolci, pronti da bere e di grande successo commerciale.                                                                                                                                                                                                                                                                                                       |
| **Vinho Verde**                      | Nel Minho, estremo nord atlantico a clima umido, nasce il Vinho Verde: bianco leggerissimo, talvolta leggermente frizzante, da uve come Alvarinho (Albariño) e Loureiro. Bassa alcolicità (9-11 %), acidità elevata e note citrine e floreali; si consuma giovane per la massima freschezza.                                                                                                                                                                                                                                                                                                                                                                                              |
| **Madeira**                          | Vino fortificato prodotto sull’isola omonima. Celebre per il processo di maderizzazione (riscaldamento controllato del vino) e per l’altissima acidità che lo rende praticamente eterno. Può essere secco o dolce, da vitigni Malvasia, Sercial, Verdelho, Boal. Aromi tipici di caramello, frutta secca tostata, iodio e curry.                                                                                                                                                                                                                                                                                                                                                          |


#### Profili Enologici di Paesi e Regioni nel Dataset: Cile

In [ ]:
summarize_feature(df_clean[df_clean.country=='Chile'], "province")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'Chile'], "province")

Considerato una “nuova frontiera” del vino dai primi anni ’90, il Cile ha guadagnato fama per l’eccellente rapporto qualità / prezzo. La viticoltura si sviluppa nelle valli trasversali che scendono dalle Ande verso l’Oceano Pacifico, con clima secco, stabile e privo di fillossera.  

| Regione / Territorio                                | Descrizione                                                                                                                                                                                                                                                                                       |
| --------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Valle del Maipo**                                 | Cuore storico della viticoltura cilena, vicino a Santiago. Qui il **Cabernet Sauvignon** è vitigno di punta: rossi strutturati, ribes nero, menta ed eucalipto hanno messo il Cile sulla mappa degli appassionati. La zona produce anche ottimi Merlot e Syrah.                                   |
| **Carménère (simbolo nazionale)**                   | Uva bordolese quasi scomparsa in patria, ha trovato in Cile la sua terra promessa: vini di colore profondo, note vegetali di peperone dolce e spezie, tannini morbidi e carattere originale; oggi identificata come “firma” del vino cileno.                                                      |
| **Valli costiere – Casablanca, Leyda, San Antonio** | Clima più fresco influenzato dal Pacifico: **Sauvignon Blanc** vibranti, ricchi di aromi tropicali e agrumati, spesso premiati per qualità a prezzo moderato; **Chardonnay** dal profilo che varia da fresco e fruttato a opulento con legno. Pinot Noir di stile elegante nelle zone più fredde. |
| **Colchagua & Cachapoal**                           | Valli interne del Rapel, celebri per rossi di grande intensità: Carménère, Cabernet Sauvignon e Syrah primeggiano, con frutto maturo, spezie e buona struttura; molti vini di punta cileni provengono da qui.                                                                                     |
| **Aconcagua & Maule**                               | Aconcagua (più a nord) e Maule (centro-sud) vantano tradizione per rossi potenti e concentrati, spesso da Cabernet, Syrah e cuvée bordolesi; il clima caldo e secco favorisce maturazioni complete. Maule ospita anche vecchie vigne di varietà tradizionali.                                     |
| **Curicó**                                          | Valle nota per produzioni affidabili da tavola e per un ampio ventaglio di vitigni internazionali; offre buoni rapporti qualità / prezzo su vini quotidiani.                                                                                                                                      |
| **Bio-Bio & Itata**                                 | Aree meridionali emergenti, più fresche e piovose: si sperimentano rossi leggeri da **País** (Mission grape) e bianchi aromatici; crescente interesse per espressioni a basso intervento e vigneti antichi a piede franco.                                                                        |


#### Profili Enologici di Paesi e Regioni nel Dataset: Argentina

In [ ]:
summarize_feature(df_clean[df_clean.country == 'Argentina'], "province")

In [ ]:
summarize_feature(df_clean[df_clean.country == 'Argentina'], "region_1")

Principale produttore sudamericano assieme al Cile, nota soprattutto per il Malbec. La viticoltura si concentra ai piedi delle Ande, in particolare nella provincia di **Mendoza** (≈ 75 % della produzione nazionale) a 600-1500 m s.l.m.; l’altitudine mitiga il clima semidesertico garantendo forte escursione termica e intensa luminosità.  

| Regione / Territorio                | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                |
| ----------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Malbec (bandiera nazionale)**     | Originario del sud-ovest francese, in Argentina ha trovato una nuova identità. I Malbec di Mendoza sono di colore violaceo impenetrabile, ricchi di prugna, mora, violetta, vaniglia e cioccolato dal rovere; al palato risultano morbidi, pieni, vellutati, con tannini dolci e acidità moderata. Zone come **Luján de Cuyo** e **Valle de Uco** producono Malbec d’eccellenza (es. Catena Zapata, Achaval Ferrer) che hanno proiettato l’Argentina sulla scena mondiale. |
| **Mendoza**                         | Cuore della viticoltura argentina; altitudini elevate, clima secco e suoli alluvionali ideali per rossi concentrati. Oltre al Malbec, si coltivano Cabernet Sauvignon (spesso in blend col Malbec per maggior struttura), Syrah e Merlot. La provincia ospita oltre 1 500 cantine e una vivace enoturismo.                                                                                                                                                                 |
| **Altri vitigni e stili**           | Tra i rossi: Cabernet Sauvignon, Syrah, Merlot e **Bonarda argentina** (Douce Noir) introdotta dagli immigrati italiani. Tra i bianchi spicca il **Torrontés** – aromatico, profumi di fiori bianchi, pesca e uva moscato ma vinificato secco – coltivato soprattutto nel Nord. Crescono anche Chardonnay e Sauvignon Blanc, in particolare ad alta quota.                                                                                                                 |
| **Salta (Cafayate)**                | Regione settentrionale con vigneti oltre 1 700-2 000 m di altitudine. Celebre per Torrontés intensi e profumati; produce anche Malbec e Cabernet con spiccata acidità e freschezza grazie all’altitudine estrema.                                                                                                                                                                                                                                                          |
| **Patagonia (Río Negro, Neuquén)**  | Zona meridionale dal clima più fresco: si ottengono Pinot Noir eleganti, bianchi tesi e alcuni rossi bordolesi dal profilo più leggero rispetto a quelli di Mendoza.                                                                                                                                                                                                                                                                                                       |

L’Argentina è famosa per vini rossi robusti e alcolici, in sintonia con la sua cucina (asado e carni alla griglia). Ha un ottimo rapporto qualità-prezzo soprattutto sul Malbec, dove bottiglie da 15-20 € possono facilmente ottenere punteggi attorno a 90/100. I top di gamma (es. monopoli di vecchie vigne) competono anch’essi a livello internazionale. L’enologia argentina negli ultimi anni sta sperimentando vendemmie più anticipate e meno legno per cercare maggiore finezza e acidità, specie con l’emergere di micro-terroir in alta quota (es. Gualtallary, Altamira in Uco Valley). 

#### Profili Enologici di Paesi e Regioni nel Dataset: Australia

In [ ]:
summarize_feature(df_clean[df_clean.country == 'Australia'], "province")

In [ ]:
summarize_feature(df_clean[df_clean.country == 'Australia'], "region_1")

Negli ultimi decenni l’Australia è diventata un attore di primo piano, capace di spaziare dai “critter wines” commerciali alle bottiglie premium ricercate. Il clima varia dal caldo desertico dell’interno a zone costiere o di altura più fresche; le aree vinicole si concentrano nel Sud del Paese (South Australia, Victoria, New South Wales, Western Australia).

| Regione / Territorio                | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
| ----------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Shiraz**                          | Vitigno simbolo. Gli Shiraz della **Barossa Valley** e di **McLaren Vale** (South Australia) sono potenti: colore violaceo denso, confettura di more, pepe nero, cioccolato, liquirizia, tannini morbidi e 14-15 % vol; icone come Penfolds Grange e Henschke Hill of Grace. Nelle zone fresche (Yarra Valley, Canberra District) lo stile diventa più pepato ed elegante, simile al Rodano settentrionale. Diffuso il blend **GSM** (Grenache-Shiraz-Mourvèdre) d’ispirazione rodigina. |
| **Cabernet Sauvignon**              | Ottimi risultati soprattutto a **Coonawarra** (South Australia) su suoli di terra rossa e a **Margaret River** (Western Australia). Vini di ribes nero intenso, eucalipto/mentolo, grande concentrazione; spesso assemblati con Merlot o piccole quote di Shiraz.                                                                                                                                                                                                                        |
| **Chardonnay**                      | Presente in molte regioni. Dopo gli anni ’90 di stile “bold” (molto legno e malolattica), oggi nelle zone fresche (Adelaide Hills, Mornington Peninsula, Tasmania) offre bianchi vibranti con pesca, agrumi e meno rovere, avvicinandosi ai modelli borgognoni; resta anche base per spumanti metodo classico.                                                                                                                                                                           |
| **Pinot Noir**                      | Nelle aree fresche del Victoria (Yarra Valley, Mornington) e in Tasmania si producono Pinot Noir fini, con ciliegia, terra umida e acidità rinfrescante; non raggiungono la complessità dei migliori Borgogna, ma la piacevolezza è in costante crescita.                                                                                                                                                                                                                                |
| **Altre produzioni**                | **Riesling** eccellente in Clare Valley ed Eden Valley (secchissimo, acidissimo, lime, kerosene con l’età); **Semillon** unico della Hunter Valley (raccolto precoce, basso alcol, alta acidità, che con l’invecchiamento sviluppa miele e tostatura); **Grenache** da vecchie vigne a McLaren Vale; **Viognier** e **Marsanne** nel Victoria; fortificati stile Porto/Sherry e l’amplissimo **Liqueur Muscat** di Rutherglen, denso come sciroppo di datteri.                           |


L’Australia vinicola è innovativa, sfrutta tecnologie avanzate e comunicazione di marketing efficace. Ha imposto nel mondo uno stile “New World” fatto di vini fruttati, corposi e consistenti. I suoi vini di massa (es. brand Jacob’s Creek, Yellow Tail) hanno avuto enorme successo globale. Nello stesso tempo, i vini di pregio australiani hanno ottenuto punteggi altissimi e collezionisti (es. Penfolds Grange ha ricevuto 100/100 in diverse annate). In termini di prezzo, l’Australia offre buoni affari in fascia media su Shiraz e blend, mentre i top di gamma sono costosi ma ancora meno dei francesi equivalenti.

#### Profili Enologici di Paesi e Regioni nel Dataset: Germania, Austria e altri stati

In [ ]:
summarize_feature(df_clean[df_clean.country == 'Germany'], "province")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'Germany'], "province")

In [ ]:
summarize_feature(df_clean[df_clean.country == 'Austria'], "province")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.country == 'Austria'], "province")

In [ ]:
summarize_feature(df_clean[df_clean.country == 'Argentina'], "province")

| Regione / Territorio   | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| ---------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Germania**           | Celebre soprattutto per i Riesling di Mosella, Rheingau, Pfalz, ecc. Stili dal secco (Trocken) ai dolci di vendemmia tardiva o ghiaccio (Eiswein), con acidità altissima e purezza aromatica. Un Mosel Spätlese può avere < 9 % vol alcol, residuo zuccherino bilanciato da acidità tagliente, profumi di fiori, lime e pietra focaia, capacità di invecchiare decenni. Ottimi anche i Pinot Nero (Spätburgunder) in Baden, Pfalz e Ahr, oltre a bianchi da Müller-Thurgau, Silvaner, Pinot Bianco/Grigio. La classificazione Prädikatswein (Kabinett, Spätlese, Auslese…) indica il tenore zuccherino del mosto. I Riesling tedeschi restano riferimento mondiale, ma i Pinot Neri stanno guadagnando consensi. |
| **Austria**            | Rinomata per il Grüner Veltliner (≈ ⅓ dei vigneti). I Grüner di Wachau, Kamptal, Kremstal sono bianchi secchi di carattere: mineralità spiccata, note di pepe bianco, mela verde e agrumi, ottimi a tavola. L’Austria eccelle anche con il Riesling (Wachau produce alcuni dei migliori secchi al mondo). Fra i rossi spiccano Blaufränkisch e Zweigelt, fruttati e pepati (soprattutto in Burgenland). Standard qualitativi alti dopo gli anni ’80; i Grüner “Smaragd” di Wachau e i dolci Beerenauslese/Trockenbeerenauslese di Illmitz ottengono punteggi top.                                                                                                                                                |
| **Nuova Zelanda**      | Famosa per il Sauvignon Blanc di Marlborough: aromi esplosivi di frutto della passione, lime, ortica, pepe verde; stile riconoscibile che ottiene punteggi elevati (Cloudy Bay fu rivoluzionario). Produce anche ottimi Pinot Noir in climi freschi (Central Otago, Martinborough), Chardonnay e un po’ di Syrah pepato a Hawke’s Bay. Viticoltura moderna, di alta qualità, seppur con produzione modesta.                                                                                                                                                                                                                                                                                                      |
| **Sud Africa**         | Storico produttore dell’emisfero sud, unisce tradizione europea a clima caldo. Vitigno identitario è il **Pinotage** (incrocio Pinot Noir × Cinsault): rossi scuri, speziati, con note affumicate e frutta nera. Più successo internazionale per Cabernet Sauvignon e Syrah di Stellenbosch e Paarl (strutturati, con note mentolate) e per eccellenti **Chenin Blanc** (Steen) secchi ma ricchi di frutto tropicale e miele. Vecchie vigne di Swartland stanno ricevendo punteggi altissimi; famoso anche il vino dolce di Constantia da Muscat de Frontignan.                                                                                                                                                  |
| **Altri paesi minori** | **Grecia**: Assyrtiko di Santorini (bianco minerale), Xinomavro tannico ma pochi campioni. **Canada**: Icewine dell’Ontario (Riesling o Vidal) dolcissimo e premiato. Presenze limitate ma arricchiscono la varietà globale.                                                                                                                                                                                                                                                                                                                                                                                        |


### 3.4.3 Sommelier

In [ ]:
summarize_feature(df_clean,"taster_name")

Circa 1/5 dei vini recensiti non presenta il nome del/dei Sommelier che ha/hanno effettuato la recensione. Provo a guardare la colonna dei nomi su twitter per osservare eventuali differenze.

In [ ]:
summarize_feature(df_clean,"taster_twitter_handle")

Le differenze, in effetti, ci sono. Se per la colonna dei "taster_name" abbiamo 24917 valori mancanti, per quella dei "taster_twitter_handle" ne abbiamo 29446. La mia intenzione è quella di verificare se posso ridurre i nomi mancanti. Per esempio, andrò a verificare se account come @worldwineguys possano essere sfruttati per questa ricerca.

In [ ]:
mask = df_clean["taster_name"].isna() & df_clean["taster_twitter_handle"].isna()
count = mask.sum()
print(str(count) + " righe con taster_name e taster_twitter_handle} entrambi NaN")

Come non detto, non abbiamo corrispondenze. Dopo un'ulteriore analisi ho appurato che @worldwineguys corrisponde ai sommelier Mike DeSimone e Jeff Jenssen. Quindi procedo a sostituire i nan con una etichetta più leggibile, come "No Taster Name".

In [ ]:
# Rimpiazza i NaN
df_clean["taster_name"].fillna("No Taster Name", inplace=True)

In [ ]:
summarize_feature(df_clean,"taster_name")

In [ ]:
# Verifico che il mapping sia corretto -> OK!
missing_counts = report_missing_values(df_clean, top_n=None, show_heatmap=True)

In [ ]:
plot_feature_distribution_interactive(df_clean, "taster_name")

Posso comunque arricchire l'analisi, ricercando la nazionalità dei Sommelier. Mi potrà essere utile più tardi.

| #  | Nome                   | Nazionalità                                                                |
| -- | ---------------------- | -------------------------------------------------------------------------- |
| 1  | **Roger Voss**         | Statunitense ([decanter.com][1])                                           |
| 2  | **Michael Schachner**  | Statunitense ([wineenthusiast.com][2])                                     |
| 3  | **Kerin O’Keefe**      | Statunitense ([en.wikipedia.org][3])                                       |
| 4  | **Paul Gregutt**       | Statunitense ([en.wikipedia.org][4])                                       |
| 5  | **Virginie Boone**     | Statunitense (nata a Parigi, vive in California) ([wineenthusiast.com][5]) |
| 6  | **Matt Kettmann**      | Statunitense ([wineenthusiast.com][6])                                     |
| 7  | **Joe Czerwinski**     | Statunitense ([winejournal.robertparker.com][7])                           |
| 8  | **Sean P. Sullivan**   | Statunitense ([wineenthusiast.com][8])                                     |
| 9  | **Anna Lee C. Iijima** | Nippo-statunitense (Japanese-American) ([newwinereview.com][9])            |
| 10 | **Jim Gordon**         | Statunitense ([wineenthusiast.com][10])                                    |
| 11 | **Anne Krebiehl MW**   | Tedesca ([decanter.com][11])                                               |
| 12 | **Lauren Buzzeo**      | Statunitense ([foodandwine.com][12])                                       |
| 13 | **Susan Kostrzewa**    | Statunitense ([traversecity.com][13])                                      |
| 14 | **Mike DeSimone**      | Statunitense ([amazon.com][14])                                            |
| 15 | **Jeff Jenssen**       | Statunitense ([amazon.com][14])                                            |
| 16 | **Alexander Peartree** | Statunitense ([muckrack.com][15])                                          |
| 17 | **Carrie Dykes**       | Statunitense ([wineenthusiast.com][16])                                    |
| 18 | **Fiona Adams**        | Statunitense ([wineenthusiast.com][17])                                    |
| 19 | **Christina Pickard**  | Statunitense ([instagram.com][18])                                         |

[1]: https://www.decanter.com/author/roger-voss "Roger Voss, Author at Decanter"
[2]: https://www.wineenthusiast.com/contributor/michael-schachner/?srsltid=AfmBOor-ncYa93L47MNjjVKAcbj3gFpYPRZB6PGcVetN8I4M5rrdEAbj "Michael Schachner | Wine Enthusiast"
[3]: https://en.wikipedia.org/wiki/Kerin_O%27Keefe "Kerin O'Keefe"
[4]: https://en.wikipedia.org/wiki/Paul_Gregutt "Paul Gregutt"
[5]: https://www.wineenthusiast.com/contributor/virginie-boone/?srsltid=AfmBOoqSlgFZir3m2wlvStPVo4DV5GfvpSkqsPvHtTmYKorB-VTjldy- "Virginie Boone | Wine Enthusiast"
[6]: https://www.wineenthusiast.com/contributor/matt-kettmann/?srsltid=AfmBOoqOkAEZAAJXT9WKyvbszY9a1ayi6_fYtS5g-FkOvdUX5q3-HBqp "Matt Kettmann | Wine Enthusiast"
[7]: https://winejournal.robertparker.com/our-contributors/51?lang=en&max=10&offset=10&order=desc&sort=publishedDate "Joe Czerwinski - Robert Parker: The Wine Advocate"
[8]: https://www.wineenthusiast.com/contributor/sean-sullivan/?srsltid=AfmBOooDJ1-ygXquXpRiri0uvwouP4fW5LAXOAne1XCadlPkrI-MuvmJ "Sean P. Sullivan - Wine Enthusiast"
[9]: https://www.newwinereview.com/author/aliijima "Anna Lee C. Iijima - The New Wine Review"
[10]: https://www.wineenthusiast.com/contributor/jim-gordon/?srsltid=AfmBOooyAuYfQGjuPrF0z94ZL557xnTpIj5hV0LP7ThbYea8Ayd14c8j "Jim Gordon | Wine Enthusiast"
[11]: https://www.decanter.com/author/anne-krebiehl "Anne Krebiehl MW, Author at Decanter"
[12]: https://www.foodandwine.com/lauren-buzzeo-8683797 "Lauren Buzzeo - Food & Wine"
[13]: https://www.traversecity.com/blog/author/susan-kostrzewa/665f7be2989f6302048b56de "Posts by Susan Kostrzewa | Traverse City Travel Blog"
[14]: https://www.amazon.com/Fire-Island-Cookbook-Jeff-Jenssen/dp/1451632932 "The Fire Island Cookbook: Jenssen, Jeff, DeSimone, Mike - Amazon.com"
[15]: https://muckrack.com/alexander-peartree "Alexander Peartree's Profile | Full Pour Journalist - Muck Rack"
[16]: https://www.wineenthusiast.com/contributor/carrie-dykes/?srsltid=AfmBOorMqaEzf4j4OgMON666WIPlaOybRl8xIZN5kaJbmW2pxvZ1ZZ68 "Carrie Dykes | Wine Enthusiast"
[17]: https://www.wineenthusiast.com/contributor/fiona-adams/?srsltid=AfmBOoohv5q7qeSOiLCjF5qPk1eV8NUsff1BAhYIqez01DjPd0Km3u_f "Fiona Adams | Wine Enthusiast"
[18]: https://www.instagram.com/p/DI_kGcotcGD/ "Christina Pickard | Brief thoughts for a short opinion piece originally ..."


### 3.4.4 Varietà

Per prima cosa, ho osservato un solo valore mancante. Controllo subito e lo rimpiazzo con un testo. 

In [ ]:
missingVariety = df_clean[df_clean['variety'].isna()]

In [ ]:
missingVariety

Dopo aver effettuato una ricerca possiamo affermare che si tratti di un Cabernet Sauvignon, pertanto aggiornerò la riga in questione

In [ ]:
# use .at when updating a single scalar (faster than .loc)
record_id = 86909
new_variety = "Cabernet Sauvignon"

df_clean.at[record_id, "variety"] = new_variety

In [ ]:
# Verifico che il mapping sia corretto -> OK!
missing_counts = report_missing_values(df_clean, top_n=None, show_heatmap=True)

#### Principali Vitigni nel Dataset

In [ ]:
summarize_feature(df_clean,"variety")[0:30]

In [ ]:
plot_feature_distribution_interactive(df_clean, "variety")

Il dataset comprende numerosi vitigni, sia internazionali che autoctoni. I più frequenti (in migliaia di recensioni) sono Pinot Noir, Chardonnay, Cabernet Sauvignon, Red Blend (blend di rossi), Bordeaux-style Red Blend, Riesling, Sauvignon Blanc, Syrah/Shiraz, Rosé, Merlot, seguiti da Nebbiolo, Zinfandel (Primitivo), Sangiovese, Malbec ecc. Ciascun vitigno ha proprie caratteristiche organolettiche e aree di elezione. Ecco una panoramica dei principali vitigni del dataset (con note degustative e diffusione geografica):

| Tipologia di vino           | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| --------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Pinot Noir** | Vitigno a bacca rossa nobile originario della Borgogna (Francia), oggi coltivato in tutto il mondo (Francia, USA – Oregon e California, Germania, Italia settentrionale, Nuova Zelanda). È un vitigno difficile e altamente sensibile al terroir. Vinificato in rosso, produce vini di colore rosso rubino tenue; profumi delicati ed eleganti di frutti rossi (lampone, ciliegia), spesso con note floreali (rosa) e speziate da giovane, che evolvono in sentori terrosi (sottobosco, tartufo) e di cuoio con l’invecchiamento. Al palato è generalmente di corpo medio, con acidità vivace, tannino fine e grande finezza. In Borgogna dà alcuni dei migliori vini rossi al mondo. Fuori dall’Europa, spiccano i Pinot Noir dell’Oregon (Willamette Valley) e della Nuova Zelanda (Marlborough), che grazie a climi simili alla Borgogna esprimono grande qualità.                                                                                                                                                                     |
| **Chardonnay**              | Vitigno a bacca bianca internazionale originario della Borgogna, diffusissimo (oltre 19 700 ha coltivati in Italia, e piantato in Europa, Americhe, Oceania). Uva estremamente versatile: produce vini bianchi di colore giallo paglierino, con aromi fruttati (spesso frutta tropicale come ananas) e floreali, sapore equilibrato e sapido. Lo Chardonnay si adatta a vari terroir sviluppando profili aromatici diversi, dai vini freschi, minerali e agrumati (es. Chablis in Borgogna) a quelli più morbidi e strutturati con note di vaniglia e frutta secca se affinati in legno. Ha acidità elevata e buon tenore alcolico, che gli permette sia di produrre eccellenti spumanti metodo classico (es. Champagne, Franciacorta) sia bianchi fermi longevi. Principali aree: Borgogna (Chardonnay di riferimento mondiale), Champagne (assemblato in molti spumanti), California (Chardonnay spesso maturati in barrique, dal gusto burroso), Italia (Trentino-Alto Adige, Friuli, Franciacorta).                                   |
| **Cabernet Sauvignon**      | Il re dei vitigni a bacca rossa internazionali, originario di Bordeaux (zona del Médoc-Graves). È un incrocio naturale tra Cabernet Franc e Sauvignon Blanc. Si è diffuso globalmente (oltre 130 000 ha nel mondo, inclusi USA, Cile, Australia, Europa) grazie alla sua adattabilità a climi diversi e alla capacità di dare vini di alta qualità e lunga longevità. Ha acini piccoli a buccia spessa, ricchi di tannini e sostanze estrattive. I vini sono di colore rubino intenso, dal profilo aromatico fruttato (note di ribes nero, cassis) ed erbaceo (peperone verde, soprattutto in climi freschi). Al palato sono caldi, di corpo, tannici e strutturati. Spesso maturano in barrique sviluppando sentori di vaniglia, spezie e cedro. In assemblaggio con Merlot e Cabernet Franc costituisce il taglio bordolese classico (es. grandi vini di Bordeaux e molti Supertuscan toscani). Zone simbolo: Bordeaux, Napa Valley, Cile (Maipo e Colchagua), Bolgheri in Toscana, Australia (Coonawarra, spesso in blend con Shiraz). |
| **Merlot**                  | Altro vitigno bordolese a bacca nera, originario della Gironda (Bordeaux). In Francia è tradizionalmente usato in assemblaggio col Cabernet Sauvignon (apporta morbidezza e frutto al taglio bordolese), ma vinificato in purezza dà anche ottimi vini. Diffuso in tutto il mondo (in Italia arrivò a fine ’800 in Friuli e Veneto). I vini sono tipicamente di colore rubino intenso, profumati di frutti rossi maturi (prugna, ciliegia, lampone) con lievi note erbacee e speziate. Al palato risultano più morbidi e rotondi rispetto al Cabernet, con tannini delicati e acidità moderata, il che li rende immediatamente piacevoli e approcciabili. I Merlot di alta qualità hanno comunque buon corpo, calore e potenziale di invecchiamento (esempi eccellenti in Bordeaux Right Bank come Pomerol e St-Émilion). In Italia è vinificato sia in purezza sia in blend (molti Supertuscan). Apprezzato per il gusto suadente e vellutato, definito “morbido e succoso”.                                                             |
| **Syrah (Shiraz)**          | Vitigno a bacca rossa originario della Valle del Rodano (Francia settentrionale), oggi coltivato con successo anche nel Nuovo Mondo (in Australia è noto come Shiraz). Produce vini dal colore rosso rubino con riflessi violacei, molto concentrati. Il bouquet è intenso e complesso: frutta scura (mora, mirtillo, prugna) e marcate note speziate di pepe nero, tabacco, liquirizia, talvolta con tocchi di cioccolato o menta. Al palato è secco, di corpo pieno, tannico e caldo, con buona acidità e persistenza. I tannini da giovani sono astringenti, ma si ammorbidiscono con l’affinamento rendendo i grandi Syrah longevi e armoniosi. Zone classiche: Rodano Nord (Hermitage, Côte-Rôtie – Syrah in purezza, eleganti e pepati), Rodano Sud (Châteauneuf-du-Pape – in blend con Grenache), Australia (Barossa Valley), Sicilia e Toscana (Syrah in purezza in alcune IGT), California (Paso Robles).                                                                                                                        |
| **Sauvignon Blanc**         | Vitigno bianco aromatico originario della Valle della Loira e di Bordeaux in Francia; il nome richiama “sauvage” (selvaggio) per gli spiccati aromi verdi e vegetali. Diffuso globalmente (Francia, Nuova Zelanda, California, Sudafrica, Cile, Italia – soprattutto Alto Adige e Friuli). I vini presentano profumi erbacei (erba tagliata, foglia di pomodoro, peperone verde, salvia, asparago) dovuti alle metossipirazine, vivaci note fruttate (agrumi, frutto della passione, kiwi, uva spina) e talvolta accenti minerali di pietra focaia. Al gusto sono molto freschi, leggero-medi di corpo e secchi. Nei climi freddi (es. Marlborough) risultano molto acidi e taglienti, nei climi caldi più pieni e tropicali. Generalmente vinificati in acciaio per preservare gli aromi varietali; l’uso del legno è raro.                                                                                                                                                                                                              |
| **Riesling**                | Nobile vitigno bianco aromatico originario della Germania (valle del Reno/Mosella). Considerato fra i migliori vitigni bianchi per eleganza e longevità, mantiene acidità elevatissima anche a piena maturazione e permette stili dal secco affilato al dolce muffato o eiswein. I vini sono profumatissimi: note floreali, agrumi, mela verde, pesca, albicocca e marcata mineralità (pietra focaia). Con l’evoluzione compaiono tipici sentori di idrocarburo (petrolio). Corpo medio-leggero, grande freschezza, spesso residuo zuccherino bilanciato dall’acidità nei climi freschi (Mosella). Coltivato con successo in Austria (Wachau, Kamptal), Francia (Alsazia), Nuovo Mondo (Australia, Washington, Finger Lakes) e in Italia (Alto Adige, Oltrepò Pavese, Friuli). I migliori Riesling possono evolvere per decenni.                                                                                                                                                                                                          |
| **Nebbiolo**                | Pregiato vitigno autoctono piemontese a bacca nera, base di celebri vini come Barolo, Barbaresco e Roero. Il nome rimanda alle nebbie autunnali delle Langhe. Uva esigente che dà il meglio in pochi terroir (Langhe, Alto Piemonte, Valtellina). I vini hanno colore granato non troppo carico, profumi finissimi di rosa appassita, viola, frutti rossi, spezie, liquirizia, erbe balsamiche, catrame e terra, soprattutto con l’invecchiamento. Descritti come “tar and roses” per il contrasto fra mineralità austera e delicata florealità. Al palato sono celebri per tannini possenti ed elevata acidità, che garantiscono corpo e longevità. Con il tempo diventano setosi e complessi. Oltre alle Langhe, è coltivato nell’Alto Piemonte (Spanna) e in Valtellina (Chiavennasca), dove produce anche lo Sforzato passito secco di grande struttura.                                                                                                                                                                              |
| **Zinfandel (Primitivo)**   | Vitigno a bacca rossa di origine croata, diffusosi in Puglia (Primitivo) e in California (Zinfandel). In California è molto coltivato per vini rossi robusti e per il rosato semi-dolce “White Zinfandel”. Dà vini rubino porpora intenso, profumati di frutta nera matura (amarena, prugna, mora) e viola; l’elevata maturazione zuccherina porta spesso a 15 % vol e sentori di confettura. L’affinamento in legno aggiunge note di cannella, cacao, pepe nero, liquirizia. Al palato è pieno, caldo, avvolgente, con tannini morbidi e buona persistenza. In Puglia spicca il Primitivo di Manduria DOC; in California si trovano versioni potenti rosse (es. Lodi) e rosé dolci leggeri. Apprezzato per immediatezza fruttata e carattere solare, ma in climi molto caldi può perdere acidità se non gestito bene.                               |
| **Sangiovese**        | Uva rossa simbolo della Toscana, cuore del Chianti Classico, del Brunello di Montalcino e del Vino Nobile di Montepulciano. Colore rubino medio-scarico, profuma di ciliegia aspra, prugna rossa, viola, erbe mediterranee e talvolta scorza d’arancia e tabacco. In bocca è di corpo medio, con acidità spiccata e tannini vivaci che gli donano un carattere sapido e gastronomico; l’affinamento in legno sviluppa note di cuoio e spezie dolci. Grazie a molteplici cloni (Brunello, Prugnolo Gentile, Sangiovese Grosso) si adatta a diversi terroir italiani, dall’Appennino romagnolo all’Umbria fino alla Sicilia; New World (California, Australia) ne dà interpretazioni più fruttate e morbide.                                                                                      |
| **Malbec**            | Vitigno bordolese quasi scomparso in patria ma rinato in Argentina, dove è divenuto icona nazionale (Mendoza, Valle de Uco). Gli acini dal pigmento intenso producono vini viola-porpora profondi, dal bouquet di prugna matura, mora, violetta, cioccolato e spezie dolci. Tannini setosi, medio-alta gradazione alcolica e fresca acidità di montagna rendono il Malbec argentino ricco ma equilibrato; in Francia (Cahors) il “Cot” è più austero, con note di grafite e maggiore rusticità. Anche Cile, Stati Uniti e Sudafrica lo coltivano con successo, offrendo versioni generose e pronte al consumo ma, nelle migliori parcelle d’altura, anche capaci di lungo invecchiamento.                                                                                                       |
| **Gewürztraminer**    | Aromatico vitigno a bacca rosa originario dell’Alsazia (e storicamente presente in Alto Adige). Sprigiona profumi prorompenti di litchi, rosa canina, zenzero, pepe bianco e frutta tropicale, talvolta con accenti di miele e incenso. Il colore è giallo dorato, la struttura piena e oleosa, l’acidità bassa-media e l’alcol spesso elevato, per un sorso vellutato e avvolgente. Può spaziare da versioni secche ma opulente a vendemmie tardive e Sélection de Grains Nobles intensamente dolci; la buccia ricca di terpeni lo rende ideale anche per macerazioni prolungate (orange wine). Coltivato pure in Germania, Nuova Zelanda e Stati Uniti, ma le espressioni più celebri restano quelle di Alsazia (Turckheim, Riquewihr) e Sudtirolo (Termeno / Tramin, da cui deriva il nome). |
| **Tempranillo**       | Vitigno rosso principe di Spagna: il nome (“temprano”, precoce) indica la maturazione anteriore rispetto ad altre varietà iberiche. Base dei grandi Rioja e Ribera del Duero, ma anche Toro e Navarra; in Portogallo è noto come Tinta Roriz/Aragonês. Vini rubino profondo con aromi di fragola, ciliegia, prugna secca, tabacco, cuoio e un tipico tocco di aneto o cocco quando affinati in rovere americano. Struttura da medio a pieno corpo, tannini saldi, acidità moderata: si presta a lunghi affinamenti, distinguendosi nelle categorie Crianza, Reserva e Gran Reserva. In climi più caldi sviluppa note di frutta nera e cioccolato; nel Nuovo Mondo (Texas, Oregon, Argentina) dà rossi corposi e speziati, pur mantenendo la sua impronta elegante e terrosa.                    |



### 3.4.5 Cantine

In [ ]:
summarize_feature(df_clean, "winery")[0:30]

In [ ]:
plot_feature_distribution_interactive(df_clean, "winery")

### 3.4.6 Prezzi

Il mercato vinicolo presenta diverse fasce di prezzo e in ciascuna di esse il consumatore si aspetta un certo livello qualitativo e stile di prodotto. È importante sottolineare che prezzo e qualità non sono sempre allineati rigidamente: si possono trovare ottimi vini a prezzi accessibili e vini costosi deludenti.

In [ ]:
summarize_feature(df_clean,"price")[0:20]

In [ ]:
plot_feature_distribution_interactive(df_clean, "price")

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.price <= 200], "price", title_prefix='Distribuzione di <=200$')

In [ ]:
plot_feature_distribution_interactive(df_clean[df_clean.price > 200], "price", title_prefix='Distribuzione di >200$')

In [ ]:
df_clean[df_clean['price'].isna()]

### 3.4.7 Punteggi

I punteggi dei vini nel dataset sono espressi su scala 100 punti, come da consuetudine delle riviste specializzate (Wine Enthusiast, Wine Spectator, Robert Parker’s Wine Advocate, etc.). In realtà, il punteggio pratico va da circa 80 a 100, poiché i vini sotto gli 80 punti sono raramente recensiti (considerati di qualità insufficiente) . Il sistema dei 100 punti fu popolarizzato da Robert Parker negli anni ‘80 e poi adottato da molte testate: semplifica la valutazione qualitativa in un numero, accompagnato però da una descrizione testuale. Generalmente i punteggi corrispondono a giudizi in
 categorie, ad esempio secondo Wine Spectator e Wine Enthusiast:

| Fascia di punteggio | Livello di qualità (descrizione WE)                                           |
| ------------------: | :---------------------------------------------------------------------------- |
|          **98–100** | **Classico** – Apice della qualità, vino straordinario in ogni aspetto.       |
|           **94–97** | **Superbo** – Eccellente realizzazione, vino di grande successo e stile.      |
|           **90–93** | **Eccellente** – Vivamente raccomandato; vino di carattere superiore.         |
|           **87–89** | **Molto Buono** – Vino ben fatto, spesso ottimo rapporto qualità/prezzo.      |
|           **83–86** | **Buono** – Vino piacevole e corretto, adatto al consumo quotidiano.          |
|           **80–82** | **Accettabile** – Vino semplice, senza difetti gravi, per contesti informali. |
|             **<80** | **Non raccomandato** – (Generalmente non pubblicato nelle recensioni).        |

N.B.: Wine Enthusiast, specificamente, non pubblica recensioni sotto 80 punti . Dunque nel dataset il range effettivo va da 80 a 100. Inoltre, punteggi da 95 in su sono rari e indicano “capolavori”; punteggi intorno a 85-88 indicano buoni vini ma non eccezionali.

Criteri di assegnazione: i punteggi vengono attribuiti tramite degustazioni alla cieca (blind tasting) in panel, per ridurre bias. I degustatori valutano diversi parametri: visivo (colore, limpidezza), olfattivo (intensità, qualità e complessità degli aromi), gustativo (struttura, equilibrio tra acidi, tannini, alcol, zuccheri, intensità e persistenza, finezza complessiva), armonia generale. Il punteggio sintetizza l’impressione qualitativa: un 90+ significa che il vino si distingue nettamente per equilibrio e complessità superiore. Ad esempio, Wine Spectator specifica che 95-100 è un vino “classic, a great wine”, 90-94 “outstanding, carattere e stile superiore”, 85-89 “very good, qualità speciale”, 80-84 “good, ben fatto” e così via .
 
Nel caso di Wine Enthusiast (la fonte del nostro dataset), le categorie hanno diciture leggermente diverse ma equivalenti come visto in tabella. Spesso nelle recensioni troviamo, accanto al numero, un breve testo che motiva il giudizio (descrivendo aromi, gusto e potenziale di evoluzione). Da notare
 che la scala 100 punti è non lineare: praticamente nessun vino scende sotto 70, e ogni soglia (es. 90) rappresenta un salto percepito importante nel prestigio. Un vino da 85 è “buono” ma ordinario, mentre da 90 in su è considerato “eccellente” e tende a essere molto ricercato dagli appassionati.

In [ ]:
summarize_feature(df_clean,"points")[0:30]

In [ ]:
plot_feature_distribution_interactive(df_clean, "points")

### 3.4.7 Designation

In [ ]:
summarize_feature(df_clean,"designation")

#### Applicare il bucketing alla colonna Designation

Il bucketing trasforma una variabile iper-frammentata in un set compatto di categorie significative, rendendo l’analisi esplorativa (e gli eventuali modelli futuri) più efficaci, leggibili e robusti.

| Problema iniziale                                                         | Effetto pratico                                                         | Bucketizzazione come soluzione                                                     |
| ------------------------------------------------------------------------- | ----------------------------------------------------------------------- | ---------------------------------------------------------------------------------- |
| **Cardinalità estrema** (\~38 000 valori distinti)                        | Grafici illeggibili, tabelle sterminate; modelli ML con matrici sparse  | Riduciamo a una ventina di etichette ⇒ dimensioni gestibili, legenda chiara        |
| **Sinonimi & lingue diverse** (*Reserve, Reserva, Riserva, Réserve*)      | Stesso concetto conta come categorie separate → diluisce le statistiche | Normalizziamo in un’unica label “Reserve”                                          |
| **Noise semantico** (cuvée di singola vigna, nomi di fantasia una tantum) | Ogni valore raro incide pochissimo ma complica l’analisi                | “Other” raccoglie la lunga coda → concentro l’attenzione sui pattern ricorrenti |
| **Missing (≈29 % NaN)**                                                   | Analisi distorte se non trattati                                        | Bucket “Unknown” permette di quantificare l’impatto dei valori mancanti            |


In [ ]:
import re
import pandas as pd
from unidecode import unidecode   # rende “Réserve” → “Reserve”

# ------------------------------------------------------------------
# Ordine di precedenza e relativi pattern
# ------------------------------------------------------------------
BUCKET_RULES = [
    # -------------------- qualità / provenienza principali --------------------
    ("Estate",          r"\bestate\b"),                             # Estate, Estate Bottled, Estate Grown
    ("Reserve",         r"\breserve\b|reserva|riserva"),            # Reserve, Réserve, Reserva, Riserva
    ("Grand_Reserve",   r"gran\s+reserve|gran\s+reserva"),          # Gran Reserva
    ("Grand_Selection", r"gran\s+selezione|grand\s+selection"),     # Gran Selezione / Grand Selection
    ("Barrel",          r"barrel\s+(sample|select)?"),              # Barrel Sample / Select
    ("Single_Vineyard", r"single\s+vineyard"),                      # Single Vineyard
    ("Old_Vine",        r"old\s+vine|vieilles\s+vignes"),           # Old Vine / Vieilles Vignes

    # -------------------- metodo spumante / stile -----------------------------
    ("Brut_Nature",     r"brut\s+nature|nature\s+brut"),            # Brut Nature
    ("Brut",            r"\bbrut\b"),                               # Brut, Brut Rosé
    ("Extra_Dry",       r"extra\s+dry"),                            # Extra Dry
    ("Rosé",            r"ros[eé]"),                                # Rosé
    ("Blanc_de_Blancs", r"blanc\s+de\s+blancs"),                    # Blanc de Blancs

    # -------------------- stile o invecchiamento ------------------------------
    ("Late_Harvest",    r"late\s+harvest|late\s+bottled"),          # Late Harvest, LBV
    ("Crianza",         r"\bcrianza\b"),                            # Crianza
    ("Vintage",         r"\bvintage\b"),                            # Vintage
    ("Classic",         r"\bclassic\b|tradition"),                  # Classic/Tradition
    ("Select",          r"\bselect\b"),                             # Generic "Select"

    # -------------------- caratteristiche di produzione -----------------------
    ("Organic",         r"organic|biologico|biodynamic"),           # Organic / Biologico / Biodynamic
    ("Unoaked",         r"\bunoaked\b"),                            # Unoaked
    ("Cuvee",           r"cuvee|cuv[ée]e"),                         # Cuvée

    # -------------------- extra --------------------------------- --------------
    
    ("Tenuta",        r"\btenuta\b"),
    ("Contrada",      r"\bcontrada\b"),
    ("Vigna",         r"\bvigna\b|vineyard"),     # cattura anche 'Vigna' IT
    ("Collection",    r"collection|collezione"),
    ("Private",       r"privat[ae]?"),            # Privat, Privata
    ("Passito",       r"passito"),                # Vini dolci siciliani
    ("DOCG/DOC",      r"\bdocg?\b|igt"),          # denominazioni
    ("Rosso",         r"\brosso\b"),
    ("Bianco",        r"\bbianco\b"),
    # -------------- NB: qualunque cosa non matchi va in "Other" --------------

]

def bucket_designation_specific(desig: str | float) -> str:
    """
    Bucketize `designation` with strict English labels.
    Precedence: Estate > Reserve > ... > Other.
    """
    if pd.isna(desig) or not isinstance(desig, str):
        return "Unknown"

    text = unidecode(desig.lower())           # remove accents, lowercase
    for bucket, pattern in BUCKET_RULES:
        if re.search(pattern, text):
            return bucket
    return "Other"                            # fallback

# ------------------------------------------------------------------
# Applico la funzione
# ------------------------------------------------------------------
df_clean["designation_bucket"] = df_clean["designation"].apply(bucket_designation_specific)

# ------------------------------------------------------------------
# Distribuzione (%) dei nuovi bucket
# ------------------------------------------------------------------
bucket_dist = (
    df_clean["designation_bucket"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
print(bucket_dist)

In [ ]:
summarize_feature(df_clean,"designation_bucket")

In [ ]:
plot_feature_distribution_interactive(df_clean, "designation_bucket")

In [ ]:
# ---------- 3. Prepare data for boxplot --------------------------
plot_df = df_clean[["designation_bucket", "points"]].dropna(subset=["points"])

# Optional: order buckets by median points
bucket_order = (
    plot_df.groupby("designation_bucket")["points"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

# ---------- 4. Create interactive boxplot ------------------------
fig = px.box(
    plot_df,
    x="designation_bucket",
    y="points",
    category_orders={"designation_bucket": bucket_order},
    title="Wine Points Distribution by Designation Bucket",
    labels={"designation_bucket": "Designation Bucket", "points": "Wine Points"},
    points="all",  # show individual points (jitter) for context
    height=600,
    width=1400,
)

fig.update_layout(xaxis_tickangle=-45)

fig

| Bucket (ordinati da sinistra)      | Osservazioni chiave                                                      | Possibile interpretazione                                                                                                              |
| ---------------------------------- | ------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------------------------------- |
| **Barrel**                         | Mediana ≈ 91; quartili stretti (90-93); diversi outlier > 95             | Campioni di botte e selezioni “Barrel Select” tendono a essere valutati solo quando la qualità è alta: abbiamo punteggi mediamente superiori. |
| **Vintage**                        | Mediana \~ 90, Q3 oltre 92; molti outlier 95-98                          | Le menzioni “Vintage” (o LBV nei Porto) indicano annate speciali che effettivamente spingono la valutazione.                           |
| **Cuvee**                          | Distribuzione simile a Vintage ma con più dispersione (IQR ampio: 87-92) | “Cuvée” è termine generico → qualità più variabile; alcuni blend spiccano, altri restano medi.                                         |
| **Estate**                         | Mediana 89-90; coda superiore fino a 97                                  | Vinificazioni da uve di proprietà (“Estate-grown / bottled”) offrono una base solida, ma non garantiscono sempre l’eccellenza.         |
| **Late\_Harvest**                  | Mediana 89; coda alta > 95; coda bassa più corta                         | Vini dolci tardivi ottengono spesso buoni score, ma sono di nicchia → campione ridotto.                                                |
| **Brut**                           | Profilo simile a Estate; coda superiore raggiunge 100                    | Spumanti “Brut” spaziano da base a prestige cuvée; ampia varianza.                                                                     |
| **Other**                          | Mediana 89-90; IQR largo                                                 | Bucket residuale: riflette lo spettro medio del data-set.                                                                              |
| **Single\_Vineyard & Old\_Vine**   | Mediane 90-91; range ristretto                                           | Micro-parcel e viti vecchie mostrano costanza di qualità.                                                                              |
| **Classic / Reserve**              | *Reserve* mediana 89; presenza di outlier 95-100                         | Non sempre “Reserve” corrisponde a punteggio elevato; term ine inflazionato rispetto a “Barrel” o “Vintage”.                           |
| **Rosé**                           | Mediana 88-89, IQR stretto; pochi punteggi altissimi                     | I rosé in genere ricevono giudizi buoni ma raramente eccezionali.                                                                      |
| **Unknown / Extra\_Dry / Unoaked** | Median e IQR più bassi (≈ 85-88)                                         | Mancanza di designazione o termini neutri (“Extra Dry”, “Unoaked”) spesso segnalano linee entry-level.                                 |


# 4.0 Analisi del Dataset

## 4.1 Fasce di Prezzo e Qualità attesa

Il mercato del vino si articola in diverse fasce di prezzo; il costo di una bottiglia riflette in genere somma dei costi di produzione (rese, legno, affinamento), brand, rarità e prestigio. Prezzo e qualità non sono però rigidamente allineati: esistono ottimi vini accessibili e vini costosi deludenti.     
| Fascia di prezzo / Categoria                                         | Descrizione                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| -------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Entry-level / Fascia bassa**<br>(≈ < 10-15 €)                      | Vini economici prodotti su larga scala, pensati per essere semplici, beverini e consumati giovani. Profili puliti ma poco complessi: es. Merlot IGT fruttato e morbido, bianchi leggeri e freschi. Comprende linee base di grandi cantine, formati brick o bag-in-box. Sotto \~5 € i compromessi qualitativi diventano marcati, pur restando prodotti corretti.                                                                                                                                                                                                                                                                                             |
| **Fascia media (“popular & premium”)**<br>(≈ 10-20 € fino a 30-40 €) | La fascia più popolosa: molte DOC/DOCG di buona qualità, etichette di medio-alto livello da regioni emergenti. A 15-20 € si trovano vini con maggiore complessità aromatica, uve migliori o affinamento più lungo; 20-40 € corrisponde al premium con vini spesso premiati (es. Chianti Classico Riserva, Barbera d’Alba di punta, entry-level Napa Cabernet, Riesling Grand Cru alsaziano). Offrono finezza, struttura e potenziale evolutivo eccellente per il prezzo.                                                                                                                                                                                    |
| **Fascia alta (“super-premium”)**<br>(≈ 50 € e oltre)                | Vini prestigiosi e rari da vigneti eccellenti e produzioni limitate, con punteggi elevati e capacità di lungo invecchiamento. Comprende Bordeaux e Borgogna blasonati, Barolo e Brunello di cru famosi, Champagne millesimati speciali, Supertuscan di punta, cult wines di Napa. Il prezzo è guidato da domanda, rarità e status; oltre 100-200 € subentrano motivazioni collezionistiche: annate mature di Bordeaux First Growth, Romanée-Conti, Supertuscan da 100/100, ecc. Il miglioramento qualitativo segue la legge dei rendimenti decrescenti: un vino da 100 € non è “due volte più buono” di uno da 50 €, ma offre sfumature in più e prestigio. |


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.graph_objects import Figure
from pandas.api.types import CategoricalDtype
from typing import Iterable, Tuple, Optional


def plot_price_quality(
    df: pd.DataFrame,
    *,
    price_col: str = "price",
    quality_col: str = "points",
    hover_cols: Optional[Iterable[str]] = None,
    price_bins: Optional[Iterable[float]] = None,
    price_labels: Optional[Iterable[str]] = None,
    log_x: bool = True,
    violin_points: str = "all",
    violin_top_n: Optional[int] = None,
    height: int = 600,
    width: int = 900,
) -> Tuple[Figure, Figure]:
    """
    Plot the relationship between **price** and **quality** for a generic dataset.

    Two interactive Plotly figures are returned:

    1. **Scatter plot** – `price_col` on *x* and `quality_col` on *y*  
       - Includes an OLS trendline (use the hover tooltip to inspect R²).  
       - `log_x` can be set to *False* for a linear scale.

    2. **Violin + box plot** – distribution of `quality_col` across custom **price bands**  
       - Bands are defined via `price_bins`/`price_labels`.  
       - Use `violin_top_n` to keep only the *n* most frequent bands.

    Parameters
    ----------
    df : pandas.DataFrame
        Source data containing at least the columns specified by `price_col`
        and `quality_col`.
    price_col : str, default ``"price"``
        Numeric column representing the product price.
    quality_col : str, default ``"points"``
        Numeric column representing quality (e.g. rating or score).
    hover_cols : iterable of str | None, default None
        Columns to appear in the Plotly hover tooltip.
        *If None*, a minimal set is used:
        ``[price_col, quality_col, "title", "winery"]`` if present in `df`.
    price_bins : iterable of float | None, default None
        Lower edges (inclusive) of price bands.
        *If None*, default edges ``[0, 15, 30, 60, 120, 250, 1000]`` are used.
    price_labels : iterable of str | None, default None
        Labels for the price bands. Must be *len(price_bins)-1*.
        *If None*, labels are auto-generated (e.g. ``<15€``, ``15-30€``, …).
    log_x : bool, default True
        Apply logarithmic scale to the *x*-axis in the scatter plot.
    violin_points : {"all", "outliers", False}, default ``"all"``
        How raw points are displayed inside each violin.  
        Mirrors the Plotly ``px.violin`` *points* argument.
    violin_top_n : int | None, default None
        If set, keeps only the *n* most frequent price bands in the violin plot.
        Useful when the number of bands is high.
    height, width : int, default (600, 900)
        Figure dimensions in pixels.

    Returns
    -------
    (plotly.graph_objects.Figure, plotly.graph_objects.Figure)
        Tuple containing *(scatter_fig, violin_fig)*.

    Notes
    -----
    * Rows with missing values in `price_col` **or** `quality_col`
      are dropped internally to avoid errors.
    * If you plan to embed the charts in a dashboard (Dash, Streamlit, etc.),
      simply omit the ``.show()`` and pass the figures directly.

    Examples
    --------
    >>> scatter, violin = plot_price_quality(
    ...     df,
    ...     hover_cols=["winery", "variety"],
    ...     log_x=False,                  # linear x-axis
    ...     price_bins=[0, 20, 50, 100, 500],
    ...     price_labels=["<20€", "20-50€", "50-100€", ">100€"],
    ... )
    >>> scatter.show()
    >>> violin.show()
    """
    # 1️⃣ Drop NaN essenziali
    data = df.dropna(subset=[price_col, quality_col]).copy()

    # 2️⃣ Hover data
    if hover_cols is None:
        default_cols = [price_col, quality_col, "title", "winery"]
        hover_cols = [c for c in default_cols if c in data.columns]
    hover_data = data[hover_cols]

    # 3️⃣ Scatter
    scatter_fig = px.scatter(
        data,
        x=price_col,
        y=quality_col,
        hover_data=hover_data,
        height=height,
        width=width,
        title=f"{price_col} vs {quality_col}",
    )
    if log_x:
        scatter_fig.update_xaxes(type="log")
    scatter_fig.update_yaxes(title=quality_col)

    # 4️⃣ Price bands (con ordine preservato)
    if price_bins is None:
        price_bins = [0, 15, 30, 60, 120, 250, 1_000]

    if price_labels is None:
        price_labels = [
            f"{price_bins[i]}-{price_bins[i + 1]}€"
            for i in range(len(price_bins) - 1)
        ]
        price_labels[0] = f"<{price_bins[1]}€"
        price_labels[-1] = f">{price_bins[-2]}€"

    # Categorical ordered per mantenere l’ordine
    cat_type = CategoricalDtype(categories=price_labels, ordered=True)
    data["price_band"] = pd.cut(
        data[price_col],
        bins=price_bins,
        labels=price_labels,
        right=False,
    ).astype(cat_type)

    # Facoltativo: tieni solo le bande più popolose
    if violin_top_n:
        top_bands = (
            data["price_band"].value_counts()
            .index[:violin_top_n]
            .tolist()
        )
        data = data[data["price_band"].isin(top_bands)]

    # 5️⃣ Violin plot con ordine esplicito
    violin_fig = px.violin(
        data,
        x="price_band",
        y=quality_col,
        box=True,
        points=violin_points,
        height=height,
        width=width,
        title=f"{quality_col} per fascia di {price_col}",
        category_orders={"price_band": price_labels},  # <-- ordine garantito
    )

    return scatter_fig, violin_fig


In [ ]:
scatter, violin = plot_price_quality(df_clean, log_x=False)
scatter.show()


### 4.1.1 Violin Plot

In [ ]:
violin.show()

In termini qualitativi, nelle fasce alte ci si aspetta equilibrio perfetto, complessità e finezza: aromi che evolvono nel bicchiere, persistenza lunghissima, armonia tra intensità e eleganza. In fascia media ci si aspetta un vino franco, ben fatto e tipico, con una buona espressione varietale/territoriale. In fascia bassa ci si accontenta di un vino corretto e piacevole, senza difetti (solforosa o volatile eccessiva, spunto acetico, ecc.) ma anche senza particolare personalità. Da considerare che il prezzo può essere influenzato anche dalla zona di produzione: ad esempio un vino base di Borgogna costerà comunque più di un vino base pugliese, per motivi di resa, costo terra e mercato, non necessariamente per qualità intrinseca superiore.

Nel dataset vediamo una rappresentazione di queste fasce: la maggioranza dei vini recensiti ha prezzo dichiarato tra circa 10 e 40 USD. 
Alcune bottiglie eccezionali toccano prezzi altissimi (fino a oltre 1000$ per rarità come il Krug Clos du Mesnil 1998 citato, o cult wines californiani), mentre altre non riportano il prezzo (valore nullo). 

È possibile raggruppare i dati in fasce di prezzo e calcolare le valutazioni medie:
ad esempio, una mia analisi sugli intervalli di prezzo mostra che i vini ottengono:

| Fascia di prezzo (USD) | Punteggio medio |
| ---------------------- | --------------- |
| Sotto 15 \$            | \~ 86 punti     |
| 15 – 30 \$             | \~ 88 punti     |
| 30 – 60 \$             | \~ 90 punti     |
| 60 – 120 \$            | \~ 92 punti     |
| 120 – 250 \$           | \~ 93 punti     |
| Oltre 250 \$           | \~ 95 punti     |


Questo conferma che, in generale, i vini costosi tendono ad avere punteggi leggermente più alti (poiché spesso sono prodotti di qualità superiore). Tuttavia, il prezzo non è sempre indicatore infallibile: come detto, esistono gemme nascoste – ad esempio vini da regioni meno conosciute o di annate minori – che offrono qualità ottima a prezzo entry (value wines), così come vini di grido sovraprezzati per moda. 

Nel mondo del vino, parte del fascino sta anche nel trovare quei vini “smart buy” che superano le aspettative della loro fascia di prezzo.

In Italia, ad esempio, secondo un’analisi Vinitaly 2025, i vini fino a 15 dollari scaffale dominano il mercato “popular”, mentre solo una piccola percentuale di prodotti si colloca oltre i 25 dollari (fascia “luxury”) indice che la gran parte dei consumatori cerca un equilibrio costo/qualità. Dunque, saper interpretare le fasce di prezzo è utile nell’analisi: può emergere, ad esempio, che certi vitigni o regioni hanno in media prezzi più elevati – segnale di posizionamento premium – mentre altri offrono opportunità a basso costo.

In [ ]:
import pandas as pd
from typing import Optional, List

def summarize_country_price_points(
    df: pd.DataFrame,
    *,
    country_col: str = "country",
    price_col: str = "price",
    points_col: str = "points",
    sort_by: str = "price_mean",
    ascending: bool = False,
    decimals: int = 2,
    dropna: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    Return a summary table with mean price and mean points per country.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame.
    country_col : str, default "country"
        Column with country names.
    price_col : str, default "price"
        Numeric column with wine prices.
    points_col : str, default "points"
        Numeric column with wine ratings / points.
    sort_by : {"price_mean", "points_mean"}, default "price_mean"
        Column to sort the summary by (descending by default).
    ascending : bool, default False
        Sort ascending or descending.
    decimals : int, default 2
        Number of decimal places in the output.
    dropna : list[str] | None, default None
        Columns to consider when dropping NaN rows before grouping.
        If None, rows with NaN in `price_col` or `points_col` are dropped.

    Returns
    -------
    pd.DataFrame
        Index   : country names  
        Columns : price_mean, points_mean
    """
    # Ensure numeric columns are numeric
    df = df.copy()
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
    df[points_col] = pd.to_numeric(df[points_col], errors="coerce")

    # Drop rows with NaN in essential columns
    if dropna is None:
        dropna = [price_col, points_col]
    data = df.dropna(subset=dropna)

    # Group by country and compute means
    summary = (
        data.groupby(country_col, dropna=False)
        .agg(
            price_mean=(price_col, "mean"),
            points_mean=(points_col, "mean"),
            _count=("record_id" if "record_id" in data.columns else price_col, "count")
        )
        .round(decimals)
        .rename(columns={"_count": "num_records"})
        .sort_values(by=sort_by, ascending=ascending)
    )

    return summary


In [ ]:
country_stats = summarize_country_price_points(df_clean)
country_stats

### 4.1.2 Visualizzazioni geografiche

In [ ]:
# Aggregazione per Paese
country = (df_clean.groupby("country")
             .agg(count=("points", "size"),
                  avg_points=("points", "mean"),
                  avg_price=("price", "mean"))
             .reset_index())

# media punti (scala RdYlGn_r: verde = top, rosso = basso)
fig_pts = px.choropleth(
    country, locations="country", locationmode="country names",
    color="avg_points",
    hover_data={"avg_points":":.2f", "count":True},
    color_continuous_scale="RdYlGn_r",
    title="Average Wine Points by Country"
)
fig_pts.update_layout(geo=dict(showframe=False, showcoastlines=False))
fig_pts.show()

# Choropleth – prezzo medio (scala OrRd per contrasto sui costi)
fig_price = px.choropleth(
    country, locations="country", locationmode="country names",
    color="avg_price",
    hover_data={"avg_price":":.2f", "count":True},
    color_continuous_scale="OrRd",
    title="Average Wine Price (USD) by Country"
)
fig_price.update_layout(geo=dict(showframe=False, showcoastlines=False))
fig_price.show()

In [ ]:
import pandas as pd
import plotly.express as px
from plotly.graph_objects import Figure
from typing import Optional, List

def correlation_heatmap(
    df: pd.DataFrame,
    *,
    numeric_only: bool = True,
    cols: Optional[List[str]] = None,
    method: str = "pearson",
    title: str = "Correlation Heatmap",
    height: int = 700,
    width: int = 800,
) -> Figure:
    """
    Build an interactive correlation heatmap using Plotly.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame.
    numeric_only : bool, default True
        If True, considers only numeric columns.
        If False, attempts to convert non-numeric columns where possible.
    cols : list[str] | None, default None
        Explicit list of columns to include. Overrides `numeric_only`.
    method : {"pearson", "spearman", "kendall"}, default "pearson"
        Correlation method used by `DataFrame.corr`.
    title : str, default "Correlation Heatmap"
        Plot title.
    height, width : int
        Figure dimensions in pixels.

    Returns
    -------
    plotly.graph_objects.Figure
        Interactive heatmap figure.
    """
    # 1️⃣ Select columns
    if cols is not None:
        data = df[cols]
    elif numeric_only:
        data = df.select_dtypes(include="number")
    else:
        data = df

    # 2️⃣ Compute correlation matrix
    corr_matrix = data.corr(method=method).round(2)

    # 3️⃣ Build Plotly heatmap
    fig = px.imshow(
        corr_matrix,
        text_auto=True,
        color_continuous_scale="RdBu_r",
        title=title,
        height=height,
        width=width,
        zmin=-1,
        zmax=1,
        aspect="auto",
    )
    fig.update_xaxes(side="top", tickangle=45)
    fig.update_yaxes(autorange="reversed")
    return fig


In [ ]:
topCountries = summarize_feature(df_clean,"country")
top10countries = topCountries[0:10].value
top10countries.to_list()

In [ ]:
# Heatmap di Correlazione Punti/Prezzo del Vino per i 10 paesi più recensiti
for country in top10countries:
    fig_corr = correlation_heatmap(df_clean[df_clean.country == country], title = "Correlation Heatmap: "+ country)
    fig_corr.show()

## 4.2 Migliori Vini e Varietà

In [ ]:
# Columns to display
cols_show = ["title", "winery", "country", "province", "variety","designation_bucket" ,"points", "price", "taster_name"]
top_n = 20

top_wines = (
    df_clean.loc[df_clean["points"].notna()]
    .sort_values(by=["points", "price"], ascending=[False, False])
    .loc[:, cols_show]
    .head(top_n)
)

In [ ]:
top_wines

### 4.2.1 Che cosa ci raccontano i primi 20 vini del dataset

| Indicatore                         | Evidenza principale                                                                                                                                                                                                                          |
| ---------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Punteggio**                      | 19 bottiglie hanno ricevuto **100/100**, 1 bottiglia (Penfolds Grange 2010) **99/100**.                                                                                                                                                      |
| **Prezzo (USD)**                   | min = 80 \$ ・ mediana ≈ 405 \$ ・ media ≈ 500 \$ ・ max = 1 500 \$.<br>Il range va da Syrah di Washington a 80 \$ fino ai Bordeaux Premier Cru del 2010 a 1 500 \$.                                                                            |
| **Paesi presenti**                 | **Francia (8)** ▸ Italia (4) ▸ Stati Uniti (4) ▸ Portogallo (2) ▸ Australia (2).                                                                                                                                                             |
| **Stili / vitigni più ricorrenti** | • **Bordeaux style red blend** (5) <br>• **Champagne / spumanti** (3) <br>• **Sangiovese/Brunello** (2) <br>• **Syrah/Shiraz** (2) <br>• **Port & fortified** (2) <br>Altri singoli: Cabernet Sauvignon, Merlot, Muscat, Portuguese Red ecc. |
| **Degustatori**                    | Roger Voss firma 8 dei 20 assaggi; seguono “No Taster Name” (6), Paul Gregutt (2) e Joe Czerwinski (2), Kerin O’Keefe (1).                                                                                                                   |

### 4.2.2 Analisi dei top 20 vini

- Otto etichette vengono dalla Francia e sei sono Bordeaux 2010: Lafite, Cheval Blanc, Haut-Brion, Léoville Las Cases e Léoville Barton (più lo Château Salon 2006). Il millesimo 2010, unanimemente celebrato in Médoc e nella Rive Droite, spiega l’eccezionale concentrazione di 100 pt. Interessante la presenza di due Champagne (Krug 2002 e Cristal 2008) accanto al blanc de blancs di Salon, prova che le grandi cuvée possono raggiungere la “perfezione” al pari dei rossi da invecchiamento.

- Quattro bottiglie italiane, tutte toscane: le due Brunello Riserva (Biondi Santi 2010 e Casanova di Neri Cerretalto 2007), il Masseto 2007 (Merlot in purezza) e il Vin Santo Occhio di Pernice 1995 di Avignonesi. Conferma del doppio volto toscano: Sangiovese tradizionale e Merlot “internazionale” coesistono ai vertici.

- Per gli U.S. abbiamo la California con un Bordeaux blend cult (Vérité 2007 La Muse, Cardinale 2006) ma anche Washington che piazza ben due Syrah a 80 dollari: Royal City e Bionic Frog. Sono le vere “value picks”: costo < 100$ a fronte di 100 pt.

- L'Australia, con un fortificato storico (Chambers Rare Muscat) e il potentissimo Penfolds Grange 2010, unico 99/100 ma al secondo prezzo più alto (850 $), piazza due bottiglie nella top20.

- Anche il Portogallo posiziona il Porto Nacional Vintage 2011 di Quinta do Noval e il Barca-Velha 2008: due simboli, uno dolce fortificato e uno dei rossi secchi più ricercati del Douro.

- Nei nostri 20 campioni la regola empirica dei punteggi/price-tier regge:

    - Oltre i 250 $  (11 vini) → punteggio medio 100, con soli due “eccezioni economiche” sotto 100 dollari.

    - 150-250 $ (4 vini) → 100 pt ma prezzi ancora “ragionevoli” per l’élite.

    - ≤ 150 $ (5 vini) → gli outlier di valore, tutti Nuovo Mondo (USA, Australia) o Bordeaux “second choices” (Léoville Barton).

- Cosa manca? Assenti i bianchi secchi non-Bordeaux, gli Champagne rosé, i Riesling, i vini spagnoli e – curiosamente – i Châteauneuf-du-Pape: segno che la selezione “perfect score” del dataset segue fortemente l’orientamento del panel (Roger Voss & Co.) e le tradizionali gerarchie del mercato primario statunitense.

Possiamo affermare che prezzo e punteggio tendenzialmente crescono insieme, ma non linearmente. Oltre una certa soglia di qualità (diciamo ~90 punti) entrano in gioco fattori di rarità e prestigio che fanno salire molto il prezzo senza che il punteggio possa salire oltre (essendo limitato a 100). 
Il dataset conferma che vini sopra $30 raramente sono scarsi, ma vini economici possono comunque sorprendere positivamente.

### 4.2.3 Migliori Vini Italiani

In [ ]:
ita_wines = df_clean.loc[df_clean['country'] == "Italy"]

top20_ita_wines = (
    ita_wines.loc[ita_wines["points"].notna()]
    .sort_values(by=["points", "price"], ascending=[False, False])
    .loc[:, cols_show]
    .head(top_n)
)

In [ ]:
plot_feature_distribution_interactive(ita_wines, "points")

In [ ]:
top20_ita_wines

### 4.2.4 Migliori Vini Siciliani e  Sardi

In [ ]:
sard_wines = ita_wines.loc[ita_wines['province'] == "Sicily & Sardinia"]

top20_srd_wines = (
    sard_wines.loc[sard_wines["points"].notna()]
    .sort_values(by=["points", "price"], ascending=[False, False])
    .loc[:, cols_show]
    .head(50)
)

In [ ]:
plot_feature_distribution_interactive(sard_wines, "points")

In [ ]:
top20_srd_wines

## 4.3 Wordcloud

Una word cloud (o “nuvola di parole”) è una rappresentazione grafica che mostra le parole più frequenti di un testo:
- ogni parola compare una sola volta;
- dimensione / spessore del carattere ↔ frequenza relativa (più grande = appare più spesso nel corpus);
- distribuzione cromatica e posizione sono in genere casuali o guidate da un algoritmo di layout, quindi non portano informazione semantica — servono solo a rendere il grafico leggibile e gradevole.

| Uso principale                           | Esempio pratico                                                    |
| ---------------------------------------- | ------------------------------------------------------------------ |
| **Scansione rapida dei temi ricorrenti** | Note di degustazione → evidenziare “blackberry”, “tobacco”, “oak”… |
| **Confronto visivo tra corpora**         | Word cloud per vini rossi vs bianchi                               |
| **Storytelling / slide**                 | Mostrare i tratti distintivi di recensioni senza tabelle numeriche |


In [ ]:
import nltk
from nltk.corpus import stopwords
import re
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [ ]:
nltk.download("stopwords")
general_sw = set(stopwords.words("english"))

In [ ]:
text_series = df_clean["description"].dropna().astype(str)  # tutte le descrizioni non-NaN
full_corpus = " ".join(text_series).lower()           # le unisce in un’unica stringa

tokens = re.findall(r"[a-z']{2,}", full_corpus)  # keep words ≥2 letters
freq = Counter(tokens)
top50 = freq.most_common(50)      # primi 50 termini più frequenti

In [ ]:
display(top50)

In [ ]:
domain_sw = {
    "wine", "wines", "flavor", "flavors", "aroma", "aromas", "palate",
    "nose", "finish", "drink", "drinking", "vintage", "aged", "aging",
    "varietal", "variety", "notes", "note", "tannins", "grape", "grapes",
    "fruit", "fruits", "juicy", "flavour", "flavours", "taste"       # GB spelling
}

Le liste standard di NLTK (o SpaCy) eliminano articoli, preposizioni, pronomi ecc. — parole che sono “rumore” in quasi qualunque task NLP.
Nel vino, però, ci sono termini altamente frequenti ma poco informativi dal punto di vista semantico:

| Esempio di parola                           | Motivo di esclusione                                             |
| ------------------------------------------- | ---------------------------------------------------------------- |
| *wine*, *wines*                             | presente in quasi tutte le descrizioni; non discrimina           |
| *flavor(s)*, *aroma(s)*, *palate*, *finish* | sono “contenitori” descrittivi, non descrivono l’aroma specifico |
| *drink*, *drinking*, *vintage*              | parole metadati/consumo, non sensoriali                          |

Se non le scartiamo, la word-cloud sarà dominata da questi termini banali, nascondendo note come blackberry, graphite, tobacco che invece differenziano i vini.

In [ ]:
wine_generic_sw = {
    # verbs
    "show", "shows", "showing", "offer", "offers", "offering",
    "give", "gives", "giving", "taste", "tastes", "tasteful",
    "come", "comes", "coming", "seem", "seems", "like",

    # fillers
    "also", "along", "well", "yet", "almost", "still", "even"
}

Dopo una prima stampa ho notato ancora del rumore. Ho creato la wine_generic_sw proprio per questo motivo.

In [ ]:
years = {str(y) for y in range(1900, 2030)}
digits = {str(d) for d in range(0, 10)}
numeric_sw = years | digits

In [ ]:
all_stopwords = general_sw | domain_sw | numeric_sw | wine_generic_sw

clean_tokens = [t for t in tokens if t not in all_stopwords]
clean_corpus = " ".join(clean_tokens)

In [ ]:
wc = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    stopwords=all_stopwords,
    max_words=150,
    colormap="Dark2",
).generate(clean_corpus)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Wine Descriptions Word-Cloud", size=10)
plt.show()

# 5.0 Sentiment Analysis

La trovi su [Kaggle](https://www.kaggle.com/code/snocco/crushing-grapes-crunching-numbers-eda-sa)!

# Grazie!

Visita il mio [blog personale](https://stefanonocco.it/) o colleghiamoci su:  
- [Kaggle](https://www.kaggle.com/snocco)
- [LinkedIn](https://www.linkedin.com/in/stefanonocco/)  
- [Instagram](https://www.instagram.com/steno.py/)  

***Stefano Nocco***